# 18 · Rétropropagation — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch18_backprop/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch18_backprop/03_notebook.ipynb`, créée par `python tools/start_chapter.py 18`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 18.Q1 à 18.Q12, 18.R1 à 18.R3, 18.1 à 18.5, 18.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 18.12 | mse_loss : la loss et son gradient | 🔨 | ★ | 10 |
|  | 18.13 | La méthode très lente : combien d'essais pour progresser ? | 🔮 | ★★ | 20 |
|  | 18.14 | Cross-entropies calculées sur les logits (binaire et softmax) | 🔨 | ★★ | 30 |
|  | 18.15 | dense_backward : les trois gradients d'une couche | 🔨 | ★★ | 25 |
|  | 18.16 | gradient_check : vérifier un gradient par différences finies | 🔨 | ★★ | 25 |
|  | 18.17 | Chasse au bug : trois gradients faux démasqués | 🐛 | ★★ | 25 |
| B | 18.18 | mlp_backward : rétropropager à travers tout le réseau | 🔨 | ★★★ | 45 |
|  | 18.19 | Entraîner le réseau 2-4-4-1 à 37 poids sur les lunes | 🔨 | ★★★ | 45 |
|  | 18.20 | La même expérience avec MLPClassifier(solver='sgd') | 📦 | ★★ | 20 |
|  | 18.21 | Reproduire les figures 18.49 à 18.57 (η = 0,5 ; 0,05 ; 0,01) | 🎨 | ★★ | 30 |
| C | 18.22 | Lire des courbes de loss : divergence, lenteur, plateau, bug | 📈 | ★★ | 20 |
|  | 18.23 | Balayage du learning rate sur cinq graines | 🔬 | ★★★ | 40 |
|  | 18.24 | Neurones saturés et ReLU mortes pendant l'entraînement | 🔬 | ★★★ | 45 |
|  | 18.25 | Écrire un test pytest de gradient pour une couche | 🛠️ | ★★ | 25 |
| D | 18.26 | Micro-autograd : la classe Value | 🔨 | ★★★ | 90 |
|  | 18.27 | À la main, Value, mlp_backward, différences finies : quatre gradients identiques ? | 🔮 | ★★ | 20 |
|  | 18.28 | MiniMLP : entraîner un petit réseau fait de Value | 🔨 | ★★★ | 45 |
|  | 18.29 | Défi MNIST : au moins 97 % avec ton MLP NumPy | 🏆 | ★★★★ | 120 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")  # your own library

## Objectifs et rappel express

- Calculer à la main les deltas, les gradients et un pas de mise à jour d'un petit réseau, avec et sans activation.
- Écrire et vérifier contre PyTorch les fonctions de `mylearn.nn.backward` (les losses avec leur gradient, `dense_backward`, `mlp_backward`, `gradient_check`) et le micro-autograd `mylearn.autograd_mini` (`Value`, `MiniMLP`).
- Entraîner un MLP from scratch et diagnostiquer le learning rate, les plateaux, la saturation et les neurones morts.

**Rappel express.** $\delta_j = \partial L / \partial z_j$ ; sortie, loss quadratique : $\delta_k = (\hat{y}_k - y_k)\,f'(z_k)$ ; caché : $\delta_j = f'(z_j) \sum_k w_{jk}\,\delta_k$ ; $\partial L / \partial w_{ij} = a_i\,\delta_j$, $\partial L / \partial b_j = \delta_j$ ; $w \leftarrow w - \eta\,\partial L / \partial w$. En matrices : $\boldsymbol{\Delta} = \mathbf{G} \odot f'(\mathbf{Z})$, $d\mathbf{W} = \mathbf{A}_{\text{prev}}^\top\boldsymbol{\Delta}$, $d\mathbf{b} = \mathbf{1}^\top\boldsymbol{\Delta}$, $\mathbf{G}_{\text{prev}} = \boldsymbol{\Delta}\mathbf{W}^\top$. Cross-entropy sur les logits : $(\mathbf{P} - \mathbf{Y})/n$ ; binaire : $(\sigma(\mathbf{z}) - \mathbf{y})/n$. Vérification : $\frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$, erreur relative $\frac{|a - n|}{\max(|a| + |n|, 10^{-12})}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 18.1 à 18.5 et 🧮 18.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un sens : `+1` (monter) ou `-1` (baisser) ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix ou une suite : les lettres collées, dans l'ordre demandé (`"CADB"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 18.6 à 18.8, la réflexion (🗣️ 18.10, 📄 18.11) et l'entretien se corrigent avec `05_solutions.md`.

**18.Q1 — Pourquoi apprendre la rétropropagation si PyTorch la fait pour toi ?**

In [ ]:
# 18.Q1: the values you found on paper
answer_18_Q1a = ...  # a) True or False
answer_18_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q1c = ...  # c) True or False

for letter, answer in zip("abc", [answer_18_Q1a, answer_18_Q1b, answer_18_Q1c]):
    wb.check(f"18.Q1{letter}", answer)

**18.Q2 — Perturber un poids au hasard : pourquoi est-ce si lent ?**

In [ ]:
# 18.Q2: the values you found on paper
answer_18_Q2a = ...  # a) a whole number
answer_18_Q2b = ...  # b) a proportion
answer_18_Q2c = ...  # c) True or False
answer_18_Q2d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_18_Q2a, answer_18_Q2b, answer_18_Q2c, answer_18_Q2d]):
    wb.check(f"18.Q2{letter}", answer)

**18.Q3 — Ce que le gradient dit à chaque poids, et le cas du biais**

In [ ]:
# 18.Q3: the values you found on paper
answer_18_Q3a = ...  # a) +1 (up) or -1 (down)
answer_18_Q3b = ...  # b) +1 (up) or -1 (down)
answer_18_Q3c = ...  # c) True or False
answer_18_Q3d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_18_Q3a, answer_18_Q3b, answer_18_Q3c, answer_18_Q3d]):
    wb.check(f"18.Q3{letter}", answer)

**18.Q4 — Le delta d'un neurone : signe et effet sur l'erreur**

In [ ]:
# 18.Q4: the values you found on paper
answer_18_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q4b = ...  # b) +1 (up) or -1 (down)
answer_18_Q4c = ...  # c) True or False
answer_18_Q4d = ...  # d) a number
answer_18_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_18_Q4a, answer_18_Q4b, answer_18_Q4c, answer_18_Q4d, answer_18_Q4e]):
    wb.check(f"18.Q4{letter}", answer)

**18.Q5 — Le réseau minuscule : couches, poids et leurs noms**

In [ ]:
# 18.Q5: the values you found on paper
answer_18_Q5a = ...  # a) a whole number
answer_18_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q5c = ...  # c) a whole number
answer_18_Q5d = ...  # d) the letter of your choice, e.g. "E"
answer_18_Q5e = ...  # e) a list of 4 numbers

for letter, answer in zip("abcde", [answer_18_Q5a, answer_18_Q5b, answer_18_Q5c, answer_18_Q5d, answer_18_Q5e]):
    wb.check(f"18.Q5{letter}", answer)

**18.Q6 — Delta de sortie, delta caché, mise à jour d'un poids**

In [ ]:
# 18.Q6: the values you found on paper
answer_18_Q6a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q6b = ...  # b) a number
answer_18_Q6c = ...  # c) a number
answer_18_Q6d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q6a, answer_18_Q6b, answer_18_Q6c, answer_18_Q6d]):
    wb.check(f"18.Q6{letter}", answer)

**18.Q7 — Rétropropager couche par couche, puis avec une activation**

In [ ]:
# 18.Q7: the values you found on paper
answer_18_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q7b = ...  # b) a whole number
answer_18_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_18_Q7d = ...  # d) a number
answer_18_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_18_Q7a, answer_18_Q7b, answer_18_Q7c, answer_18_Q7d, answer_18_Q7e]):
    wb.check(f"18.Q7{letter}", answer)

**18.Q8 — Learning rate trop grand ou trop petit : reconnaître les symptômes**

In [ ]:
# 18.Q8: the values you found on paper
answer_18_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_18_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q8c = ...  # c) True or False
answer_18_Q8d = ...  # d) the letter of your choice, e.g. "E"
answer_18_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_18_Q8a, answer_18_Q8b, answer_18_Q8c, answer_18_Q8d, answer_18_Q8e]):
    wb.check(f"18.Q8{letter}", answer)

**18.Q9 — Ce que la rétropropagation propage vraiment**

In [ ]:
# 18.Q9: the values you found on paper
answer_18_Q9a = ...  # a) True or False
answer_18_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q9c = ...  # c) four letters in order, e.g. "CADB"
answer_18_Q9d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q9a, answer_18_Q9b, answer_18_Q9c, answer_18_Q9d]):
    wb.check(f"18.Q9{letter}", answer)

**18.Q10 — Neurones saturés et ReLU mortes**

In [ ]:
# 18.Q10: the values you found on paper
answer_18_Q10a = ...  # a) True or False
answer_18_Q10b = ...  # b) a number
answer_18_Q10c = ...  # c) letters glued in alphabetical order, e.g. "BD"
answer_18_Q10d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q10a, answer_18_Q10b, answer_18_Q10c, answer_18_Q10d]):
    wb.check(f"18.Q10{letter}", answer)

**18.Q11 — Mini-batches, mises à jour parallèles et coût du backward**

In [ ]:
# 18.Q11: the values you found on paper
answer_18_Q11a = ...  # a) a whole number
answer_18_Q11b = ...  # b) True or False
answer_18_Q11c = ...  # c) the letter of your choice, e.g. "E"
answer_18_Q11d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_Q11a, answer_18_Q11b, answer_18_Q11c, answer_18_Q11d]):
    wb.check(f"18.Q11{letter}", answer)

**18.Q12 — Aucune garantie, une longue histoire, des équations à apprivoiser**

In [ ]:
# 18.Q12: the values you found on paper
answer_18_Q12a = ...  # a) True or False
answer_18_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_18_Q12c = ...  # c) a year
answer_18_Q12d = ...  # d) a year
answer_18_Q12e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_18_Q12a, answer_18_Q12b, answer_18_Q12c, answer_18_Q12d, answer_18_Q12e]):
    wb.check(f"18.Q12{letter}", answer)

**18.R1 — La dérivée de la sigmoïde à partir de sa sortie**

In [ ]:
# 18.R1: the values you found on paper
answer_18_R1a = ...  # a) a number
answer_18_R1b = ...  # b) a number
answer_18_R1c = ...  # c) a number
answer_18_R1d = ...  # d) True or False
answer_18_R1e = ...  # e) a number

for letter, answer in zip("abcde", [answer_18_R1a, answer_18_R1b, answer_18_R1c, answer_18_R1d, answer_18_R1e]):
    wb.check(f"18.R1{letter}", answer)

**18.R2 — Chercher le meilleur learning rate par recherche en grille**

In [ ]:
# 18.R2: the values you found on paper
answer_18_R2a = ...  # a) a whole number
answer_18_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_18_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_18_R2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_R2a, answer_18_R2b, answer_18_R2c, answer_18_R2d]):
    wb.check(f"18.R2{letter}", answer)

**18.R3 — Représentation, évaluation, optimisation dans un MLP**

In [ ]:
# 18.R3: the values you found on paper
answer_18_R3a = ...  # a) the letter of your choice, e.g. "E"
answer_18_R3b = ...  # b) the letter of your choice, e.g. "E"
answer_18_R3c = ...  # c) the letter of your choice, e.g. "E"
answer_18_R3d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_18_R3a, answer_18_R3b, answer_18_R3c, answer_18_R3d]):
    wb.check(f"18.R3{letter}", answer)

**Ex 18.1 — Le delta, amplificateur de changement**

In [ ]:
# Ex 18.1: the values you found on paper
answer_18_1a = ...  # a) a number
answer_18_1b = ...  # b) a number
answer_18_1c = ...  # c) a number
answer_18_1d = ...  # d) a number
answer_18_1e = ...  # e) a number
answer_18_1f = ...  # f) a number with 2 decimals
answer_18_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_18_1a, answer_18_1b, answer_18_1c, answer_18_1d, answer_18_1e, answer_18_1f, answer_18_1g]):
    wb.check(f"18.1{letter}", answer)

**Ex 18.2 — Deltas de sortie avec l'erreur quadratique : attention au signe**

In [ ]:
# Ex 18.2: the values you found on paper
answer_18_2a = ...  # a) a number
answer_18_2b = ...  # b) a number
answer_18_2c = ...  # c) a number
answer_18_2d = ...  # d) a number
answer_18_2e = ...  # e) a number
answer_18_2f = ...  # f) a number
answer_18_2g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_18_2a, answer_18_2b, answer_18_2c, answer_18_2d, answer_18_2e, answer_18_2f, answer_18_2g]):
    wb.check(f"18.2{letter}", answer)

**Ex 18.3 — Descente sur une parabole : quel learning rate ?**

In [ ]:
# Ex 18.3: the values you found on paper
answer_18_3a = ...  # a) a number
answer_18_3b = ...  # b) a number
answer_18_3c = ...  # c) a number with 3 decimals
answer_18_3d = ...  # d) a number
answer_18_3e = ...  # e) a number
answer_18_3f = ...  # f) a number
answer_18_3g = ...  # g) a number
answer_18_3h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_18_3a, answer_18_3b, answer_18_3c, answer_18_3d, answer_18_3e, answer_18_3f, answer_18_3g, answer_18_3h]):
    wb.check(f"18.3{letter}", answer)

**Ex 18.4 — Rétropropagation complète du réseau minuscule, sans activation**

In [ ]:
# Ex 18.4: the values you found on paper
answer_18_4a = ...  # a) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4b = ...  # b) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4c = ...  # c) a number
answer_18_4d = ...  # d) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4e = ...  # e) a list of 2 numbers, e.g. [1.5, -2]
answer_18_4f = ...  # f) a list of 4 numbers
answer_18_4g = ...  # g) a list of 4 numbers
answer_18_4h = ...  # h) a number
answer_18_4i = ...  # i) a number with 4 decimals

for letter, answer in zip("abcdefghi", [answer_18_4a, answer_18_4b, answer_18_4c, answer_18_4d, answer_18_4e, answer_18_4f, answer_18_4g, answer_18_4h, answer_18_4i]):
    wb.check(f"18.4{letter}", answer)

**Ex 18.5 — Le même réseau avec une sigmoïde cachée : le facteur σ'(z)**

In [ ]:
# Ex 18.5: the values you found on paper
answer_18_5a = ...  # a) a number with 4 decimals
answer_18_5b = ...  # b) a number with 4 decimals
answer_18_5c = ...  # c) a number with 4 decimals
answer_18_5d = ...  # d) a number with 4 decimals
answer_18_5e = ...  # e) a number with 4 decimals
answer_18_5f = ...  # f) a number with 4 decimals
answer_18_5g = ...  # g) a number with 4 decimals
answer_18_5h = ...  # h) a number with 4 decimals

for letter, answer in zip("abcdefgh", [answer_18_5a, answer_18_5b, answer_18_5c, answer_18_5d, answer_18_5e, answer_18_5f, answer_18_5g, answer_18_5h]):
    wb.check(f"18.5{letter}", answer)

**Ex 18.9 — Fermi : rétropropagation contre force brute**

In [ ]:
# Ex 18.9: the values you found on paper
answer_18_9a = ...  # a) a whole number
answer_18_9b = ...  # b) a whole number of thousands
answer_18_9c = ...  # c) a whole number
answer_18_9d = ...  # d) a whole number of thousands
answer_18_9e = ...  # e) a whole number of milliseconds
answer_18_9f = ...  # f) a number of hours, 1 decimal

for letter, answer in zip("abcdef", [answer_18_9a, answer_18_9b, answer_18_9c, answer_18_9d, answer_18_9e, answer_18_9f]):
    wb.check(f"18.9{letter}", answer)

## Partie A · Les losses et la passe arrière d'une couche

Écrire les losses de ta librairie avec leur gradient (18.12, 18.14), mesurer ce que coûte vraiment la méthode très lente du livre (18.13), écrire la passe arrière d'une couche dense (18.15), puis l'outil qui vérifie tout gradient (18.16) et t'en servir pour démasquer trois gradients faux (18.17). La cellule suivante définit les outils des parties A à D. Ne recopie jamais un nombre à la main : chaque réponse est calculée par ton code, et pas arrondie.

In [ ]:
# Tools for the notebook exercises (parts A to D)
import contextlib
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy import special
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from wb.errors import MylearnMissingError

try:
    import torch
except ImportError:                       # the comparisons with PyTorch are then left to the tests
    torch = None


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def print_answer(ex_id, value, **_):
    """Solutions notebook: show the value that the exercise notebook checks with wb.check."""
    try:
        shown = np.round(np.asarray(value, dtype=float), 4).tolist() if isinstance(value, (np.ndarray, list, tuple)) else value
    except (TypeError, ValueError):
        shown = value
    print(f"{ex_id}:", shown)


@contextlib.contextmanager
def checked(ex_id):
    """Like wb.attempt (⏳ while the code is not written yet), but any other error raised by YOUR code becomes one ❌
    line instead of stopping the cell, so that the tests run after it can say more."""
    with wb.attempt(ex_id):
        try:
            yield
        except (NotImplementedError, ImportError, MylearnMissingError):
            raise
        except Exception as exc:                              # a bug in the learner's code: report it, go on
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                         # wb.attempt explains the forgotten return
            print(f"❌ Ex {ex_id} : ton code lève {type(exc).__name__} : {str(exc)[:300]}")


def written(probe):
    """False while probe() stops on code not written yet (NotImplementedError, or no mylearn at all); True otherwise,
    even when it raises another error (the tests then say which)."""
    try:
        probe()
    except (NotImplementedError, ImportError, MylearnMissingError):
        return False
    except Exception:
        return True
    return True


def same(ex_id, label, got, want, rtol=1e-9, atol=1e-12):
    """✅ if `got` is an array equal to `want` up to rounding, ❌ with both values otherwise; returns the verdict."""
    if got is None:
        print(f"❌ Ex {ex_id} : {label} renvoie None : as-tu oublié le return ?")
        return False
    try:
        arr = np.asarray(got, dtype=float)
    except (TypeError, ValueError):
        print(f"❌ Ex {ex_id} : {label} : attendu un tableau de nombres, reçu un objet {type(got).__name__}.")
        return False
    want = np.asarray(want, dtype=float)
    if arr.shape != want.shape:
        print(f"❌ Ex {ex_id} : {label} : attendu la forme {want.shape}, reçu la forme {arr.shape}.")
        return False
    ok = bool(np.allclose(arr, want, rtol=rtol, atol=atol, equal_nan=True))
    shown = np.round(arr, 8).tolist() if arr.size <= 6 else f"un tableau {arr.shape}"
    verdict(ex_id, ok, f"{label} : {shown}, comme attendu.",
            f"{label} : attendu {np.round(want, 8).tolist() if want.size <= 6 else 'autre chose'}, reçu {shown}"
            + ("" if want.size <= 6 else f" (plus grand écart {np.nanmax(np.abs(arr - want)):.3g})") + ".")
    return ok


def pair(ex_id, label, result):
    """Unpack a (loss, grad) result: (float, array) or None with a ❌ line when the shape of the result is wrong."""
    if result is None:
        print(f"❌ Ex {ex_id} : {label} renvoie None : as-tu oublié le return ?")
        return None
    if not isinstance(result, tuple) or len(result) != 2:
        print(f"❌ Ex {ex_id} : {label} doit rendre un tuple (loss, grad), reçu {type(result).__name__}.")
        return None
    loss, grad = result
    if not isinstance(loss, float):
        print(f"❌ Ex {ex_id} : {label} : la loss doit être un float Python, reçu {type(loss).__name__} "
              "(convertis avec float(...)).")
        return None
    return loss, np.asarray(grad, dtype=float)


def refuses(ex_id, label, call, error=ValueError):
    """✅ if call() raises `error`, ❌ otherwise (the error raised, or nothing)."""
    try:
        call()
    except error as exc:
        print(f"✅ Ex {ex_id} : {label} : {error.__name__} levée ({str(exc)[:120]}).")
        return True
    except (NotImplementedError, MylearnMissingError):
        raise
    except Exception as exc:
        print(f"❌ Ex {ex_id} : {label} : attendu {error.__name__}, reçu {type(exc).__name__} : {str(exc)[:160]}")
        return False
    print(f"❌ Ex {ex_id} : {label} : attendu {error.__name__}, mais rien n'a été levé.")
    return False


def numeric_grad(f, array, eps=1e-6):
    """Central finite differences of the scalar function f() with respect to every entry of `array` (changed in place,
    then restored). The oracle of the checks: it does not use your gradient_check."""
    grad = np.zeros_like(array, dtype=float)
    for idx in np.ndindex(array.shape):
        old = array[idx]
        array[idx] = old + eps
        plus = f()
        array[idx] = old - eps
        minus = f()
        array[idx] = old
        grad[idx] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    """max |a - b| / max(|a| + |b|, 1e-12), entry by entry (the measure of gradient_check)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.max(np.abs(a - b) / np.maximum(np.abs(a) + np.abs(b), 1e-12)))


def moons_18(n=1500, seed=1819):
    """The moons of the book's experiment (§18.11.1): about 1 500 points, standardized."""
    X, y = wb.synth.make_moons(n=n, noise=0.1, seed=seed)
    return (X - X.mean(axis=0)) / X.std(axis=0), y


TEST_FILES = {"backward": "tests/test_ch18_backward.py", "autograd_mini": "tests/test_ch18_autograd_mini.py"}


def run_mylearn_tests(module, keyword, impl="learner"):
    """Run the tests of mylearn (nn/backward.py or autograd_mini.py) selected by `keyword` (on YOUR code by default)."""
    test_file = TEST_FILES[module]
    command = [sys.executable, "-m", "pytest", test_file, "-k", keyword, "-q", "-p", "no:cacheprovider",
               "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix(f"FAILED {test_file}::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


print("tools ready · PyTorch", "available" if torch is not None else "absent (the tests will compare with SciPy)")

### Ex 18.12 — mse_loss : la loss et son gradient 🔨 ★ ⏱️ 10 min
**Objectif :** écrire l'erreur quadratique moyenne et son gradient, sans diffusion entre prédiction et cible.  
**Prérequis :** ch. 9 (erreur quadratique moyenne) · fiche §18.6, section sur les losses calculées sur les logits · **Fil rouge :** synth · **mylearn :** `nn/backward.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/backward.py` (créé par `python tools/start_chapter.py 18`), lis la docstring de la fonction, remplace le `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement ; PyTorch et SciPy sont les **oracles** des tests (sans PyTorch, les tests qui en ont besoin sont ignorés). La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à des calculs indépendants, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch18_backward.py -q` les lance tous dans un terminal.

**Contrat commun des losses** (les docstrings, plus ces précisions que vérifient les tests) : chaque loss rend un couple `(loss, grad)`, la loss **moyenne** du batch en un `float` Python, en nats (logarithme népérien), et son gradient par rapport au premier argument dans un **nouveau** tableau `float64` de la même forme ; les entrées (des listes sont acceptées) ne sont jamais modifiées ; **aucune diffusion** entre prédiction et cible : des formes différentes, même `(3,)` contre `(3, 1)`, lèvent `ValueError`.

Écris `mse_loss(y_pred, y_true)`. **Contrat** propre à cette loss : la moyenne porte sur **toutes** les cases du tableau, $L = \frac{1}{m}\sum (\hat{y} - y)^2$ avec $m$ le nombre de cases (et non le nombre de lignes), et le gradient vaut $\frac{2}{m}(\hat{\mathbf{y}} - \mathbf{y})$, comme `torch.nn.functional.mse_loss` avec `reduction="mean"`.

La vérification essaie l'exemple de la docstring, compare la loss et le gradient d'un tableau $(4, 3)$ à NumPy, vérifie que les entrées sont intactes et qu'une prédiction $(3,)$ contre une cible $(3, 1)$ est refusée, puis lance les tests de l'exercice (`-k test_mse_loss_`).

Dans tes notes : que calculerait `np.mean((y_pred - y_true) ** 2)` pour une prédiction de forme $(3,)$ et une cible de forme $(3, 1)$, si l'on ne vérifiait pas les formes ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")   # reload your saved file
with checked("18.12"):
    B = mylearn.nn.backward
    got_12 = pair("18.12", "mse_loss([1, 2], [0, 0])", B.mse_loss([1.0, 2.0], [0.0, 0.0]))
    if got_12 is not None:
        verdict("18.12", abs(got_12[0] - 2.5) < 1e-12, "mse_loss([1, 2], [0, 0]) : loss 2,5, comme la docstring.",
                f"mse_loss([1, 2], [0, 0]) : loss {got_12[0]!r}, attendu 2,5 (la moyenne de 1² et 2²).")
        same("18.12", "son gradient", got_12[1], [1.0, 2.0])
with checked("18.12"):
    B = mylearn.nn.backward
    rng_12 = np.random.default_rng(1812)
    pred_12, target_12 = rng_12.normal(size=(4, 3)), rng_12.normal(size=(4, 3))
    before_12 = (pred_12.copy(), target_12.copy())
    got_12 = pair("18.12", "mse_loss sur un tableau (4, 3)", B.mse_loss(pred_12, target_12))
    if got_12 is not None:
        same("18.12", "la loss d'un tableau (4, 3), comme np.mean((ŷ - y) ** 2)", got_12[0], np.mean((pred_12 - target_12) ** 2))
        same("18.12", "le gradient d'un tableau (4, 3), comme 2 (ŷ - y) / 12", got_12[1], 2 * (pred_12 - target_12) / 12)
        verdict("18.12", np.array_equal(pred_12, before_12[0]) and np.array_equal(target_12, before_12[1]),
                "les entrées sont intactes.", "mse_loss a modifié une de ses entrées : calcule dans de nouveaux tableaux.")
    refuses("18.12", "une prédiction (3,) et une cible (3, 1)", lambda: B.mse_loss(np.zeros(3), np.zeros((3, 1))))
if written(lambda: mylearn.nn.backward.mse_loss([1.0], [0.0])):
    run_mylearn_tests("backward", keyword="test_mse_loss_")
else:
    print("⏳ Ex 18.12 : les tests attendent ta fonction.")

### Ex 18.13 — La méthode très lente : combien d'essais pour progresser ? 🔮 ★★ ⏱️ 20 min
**Objectif :** mesurer ce que coûte la méthode « essayer un poids au hasard » du livre, face à la descente de gradient, sur un petit et sur un gros réseau.  
**Prérequis :** ch. 16 (passe avant d'un MLP) · fiche §18.2.1, §18.2.2 · **Fil rouge :** synth · **Parcours :** C

Sur les 1 500 points de lunes standardisés du livre (§18.11.1), deux réseaux à couches cachées ReLU et sortie linéaire (le logit), entraînés avec la cross-entropy binaire : le réseau du livre, 2-4-4-1 (37 paramètres), et un réseau plus gros, 2-32-32-1 (1 185 paramètres). La cellule suivante définit la **méthode très lente** du §18.2.1 (`slow_method_13` : à chaque essai, un seul paramètre tiré au hasard change de $m \sim \mathcal{N}(0 ;\ 0{,}1^2)$, et le changement n'est gardé que si la loss de tout l'entraînement baisse : une passe avant par essai), et la descente de gradient sur tout le batch (`gradient_descent_13`, $\eta = 0{,}5$ : un pas coûte environ trois passes avant). Les deux partent des mêmes poids.

Prédis, **avant** d'exécuter l'expérience :
a) `accepted_13` : pour le réseau 2-4-4-1, la part des 3 000 essais acceptés : (A) moins de 25 % ; (B) de 25 % à 45 % ; (C) de 45 % à 55 % ; (D) plus de 55 % ;
b) `small_beats_13` : pour le réseau 2-4-4-1, la loss après 3 000 essais de la méthode lente est-elle plus basse que la loss après 30 pas de gradient ? (`True` ou `False`) ;
c) `ratio_13` : pour le réseau 2-32-32-1, à coût à peu près égal (3 000 essais contre 1 000 pas de gradient), la loss de la méthode lente divisée par celle de la descente de gradient : (A) moins de 1 ; (B) de 1 à 10 ; (C) de 10 à 100 ; (D) plus de 100.

Puis exécute l'expérience : elle affiche les losses, trace les deux méthodes en fonction de leur coût en passes avant, et vérifie tes prédictions.

In [ ]:
X_13, y_13 = moons_18()
Y_13 = y_13[:, None].astype(float)


def init_13(sizes, seed):
    """He initialization (chapter 16), biases at 0: the list of [W, b] of an MLP with ReLU hidden layers."""
    rng = np.random.default_rng(seed)
    return [[rng.normal(0.0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in zip(sizes[:-1], sizes[1:])]


def forward_13(params):
    """Forward pass on the 1 500 moons: ReLU hidden layers, a linear output (the logit); returns (logit, cache)."""
    a, cache = X_13, []
    for layer, (W, b) in enumerate(params):
        z = a @ W + b
        cache.append((a, z))
        a = np.maximum(z, 0.0) if layer < len(params) - 1 else z
    return a, cache


def loss_13(params):
    """Mean binary cross-entropy computed on the logits (one forward pass)."""
    z, _ = forward_13(params)
    return float(np.mean(np.maximum(z, 0) - Y_13 * z + np.log1p(np.exp(-np.abs(z)))))


def gradient_13(params):
    """The gradient of loss_13 by backpropagation (written for you here; you will write it in 18.18)."""
    z, cache = forward_13(params)
    g = (special.expit(z) - Y_13) / len(X_13)
    grads = [None] * len(params)
    for layer in range(len(params) - 1, -1, -1):
        a, z_l = cache[layer]
        delta = g if layer == len(params) - 1 else g * (z_l > 0)
        grads[layer] = (a.T @ delta, delta.sum(axis=0))
        g = delta @ params[layer][0].T
    return grads


def slow_method_13(params, trials, seed, scale=0.1):
    """The very slow method of §18.2.1: change ONE random weight (or bias) by m ~ N(0, scale**2), keep it if the loss
    went down. Returns (fraction of accepted trials, loss after each trial)."""
    rng = np.random.default_rng(seed)
    params = [[W.copy(), b.copy()] for W, b in params]
    arrays = [array for layer in params for array in layer]
    total = sum(array.size for array in arrays)
    current, accepted, history = loss_13(params), 0, []
    for _ in range(trials):
        k = int(rng.integers(total))
        for array in arrays:                        # find the array that holds the k-th parameter
            if k < array.size:
                break
            k -= array.size
        flat = array.reshape(-1)
        old = flat[k]
        flat[k] = old + rng.normal(0.0, scale)
        new = loss_13(params)                       # one forward pass per trial
        if new < current:
            current, accepted = new, accepted + 1
        else:
            flat[k] = old
        history.append(current)
    return accepted / trials, history


def gradient_descent_13(params, steps, lr=0.5):
    """Full-batch gradient descent (each step: one forward and one backward pass, about three forward passes)."""
    params = [[W.copy(), b.copy()] for W, b in params]
    history = [loss_13(params)]
    for _ in range(steps):
        grads = gradient_13(params)
        params = [[W - lr * dW, b - lr * db] for (W, b), (dW, db) in zip(params, grads)]
        history.append(loss_13(params))
    return history


SMALL_13, LARGE_13 = [2, 4, 4, 1], [2, 32, 32, 1]
print("parameters:", {str(s): sum(a * b + b for a, b in zip(s[:-1], s[1:])) for s in (SMALL_13, LARGE_13)})

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
accepted_13 = ...      # a) "A", "B", "C" or "D"
small_beats_13 = ...   # b) True or False
ratio_13 = ...         # c) "A", "B", "C" or "D"

In [ ]:
if any(answer is ... or answer is None for answer in [accepted_13, small_beats_13, ratio_13]):
    print("⏳ Ex 18.13 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    results_13 = {}
    for sizes_13 in (SMALL_13, LARGE_13):
        start_13 = init_13(sizes_13, seed=1813)
        rate_13, slow_curve_13 = slow_method_13(start_13, trials=3000, seed=18131)
        gd_curve_13 = gradient_descent_13(start_13, steps=1000)
        results_13[str(sizes_13)] = (rate_13, slow_curve_13, gd_curve_13)
        print(f"{str(sizes_13):14s} accepted {rate_13:.1%} · loss at start {gd_curve_13[0]:.4f} · slow, 3 000 trials "
              f"{slow_curve_13[-1]:.4f} · 30 gradient steps {gd_curve_13[30]:.4f} · 1 000 gradient steps {gd_curve_13[1000]:.4f}")
    fig_13, axes_13 = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    for ax, (name_13, (rate_13, slow_curve_13, gd_curve_13)) in zip(axes_13, results_13.items()):
        ax.plot(np.arange(1, 3001), slow_curve_13, label="méthode lente (1 passe avant par essai)")
        ax.plot(3 * np.arange(1001), gd_curve_13, label="descente de gradient (≈ 3 passes par pas)")
        ax.set_yscale("log")
        ax.set_xlabel("coût, en passes avant")
        ax.set_title(f"réseau {name_13}")
        ax.legend()
    axes_13[0].set_ylabel("loss d'entraînement (échelle log)")
    plt.tight_layout()
    plt.show()
    wb.check("18.13a", accepted_13)
    wb.check("18.13b", small_beats_13)
    wb.check("18.13c", ratio_13)

Dans tes notes : pourquoi la part d'essais acceptés s'éloigne-t-elle de ce que donnerait un simple pile ou face sur le sens du changement ? Combien d'essais faudrait-il, au moins, pour toucher une fois chacun des 1 185 paramètres du gros réseau, et pour un réseau d'un million de paramètres ? Qu'est-ce qui, dans le coût d'un pas de gradient, ne dépend pas du nombre de paramètres ?

### Ex 18.14 — Cross-entropies calculées sur les logits (binaire et softmax) 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire les deux cross-entropies directement sur les logits, sans débordement, avec leur gradient si simple.  
**Prérequis :** Ex 18.12 · ch. 17 (`sigmoid`, `log_softmax`) · ch. 6 (cross-entropy) · fiche, section sur les losses calculées sur les logits · **Fil rouge :** synth · **mylearn :** `nn/backward.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/backward.py` (créé par `python tools/start_chapter.py 18`), lis la docstring de la fonction, remplace le `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement ; PyTorch et SciPy sont les **oracles** des tests (sans PyTorch, les tests qui en ont besoin sont ignorés). La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à des calculs indépendants, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch18_backward.py -q` les lance tous dans un terminal.

**Contrat commun des losses** (les docstrings, plus ces précisions que vérifient les tests) : chaque loss rend un couple `(loss, grad)`, la loss **moyenne** du batch en un `float` Python, en nats (logarithme népérien), et son gradient par rapport au premier argument dans un **nouveau** tableau `float64` de la même forme ; les entrées (des listes sont acceptées) ne sont jamais modifiées ; **aucune diffusion** entre prédiction et cible : des formes différentes, même `(3,)` contre `(3, 1)`, lèvent `ValueError`.

Écris `binary_cross_entropy_with_logits(logits, y_true)` et `softmax_cross_entropy(logits, y_true)`. **Contrat** propre à ces losses :
- la cross-entropy binaire accepte des logits de forme $(n,)$ ou $(n, 1)$ et des cibles de la même forme, **dans** $[0, 1]$ (une cible hors de cet intervalle lève `ValueError`) ; elle reste finie, sans `RuntimeWarning`, pour des logits de $\pm 1\,000$ ; son gradient vaut $(\sigma(\mathbf{z}) - \mathbf{y})/n$ ;
- la cross-entropy softmax reçoit des logits de forme $(n, K)$ et, au choix, des **labels** entiers de forme $(n,)$ (des flottants entiers comme `2.0` sont acceptés ; un label égal à $K$ ou à $-1$ lève `ValueError`, car NumPy prendrait sans rien dire la dernière colonne pour $-1$) ou des **probabilités cibles** de forme $(n, K)$ (une cible $(1, K)$, qui se diffuserait, est refusée) ; elle reste finie pour des logits de $10^4$ ; son gradient vaut $(\mathbf{P} - \mathbf{Y})/n$, avec $\mathbf{Y}$ le one-hot des labels ou les probabilités données.

Tu peux réutiliser `sigmoid` et `log_softmax` de ton `nn/activations.py` du ch. 17 (`from .activations import sigmoid, log_softmax` ; s'il te manque, la référence le remplace).

La vérification essaie les exemples des docstrings, compare les losses et les gradients à des calculs SciPy, essaie des logits extrêmes et des cibles invalides, puis lance les tests des deux fonctions.

Dans tes notes : pour $z = 40$ et une cible $y = 0$, que donne le calcul naïf $-\log(1 - \sigma(z))$ en `float64` ? Et la forme stable de la fiche ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")   # reload your saved file
with checked("18.14"):
    B = mylearn.nn.backward
    got_14 = pair("18.14", "binary_cross_entropy_with_logits([0, 0], [1, 0])",
                  B.binary_cross_entropy_with_logits([0.0, 0.0], [1.0, 0.0]))
    if got_14 is not None:
        same("18.14", "sa loss (log 2)", got_14[0], np.log(2.0))
        same("18.14", "son gradient", got_14[1], [-0.25, 0.25])
    got_14 = pair("18.14", "softmax_cross_entropy([[0, 0]], [[0.25, 0.75]])",
                  B.softmax_cross_entropy([[0.0, 0.0]], [[0.25, 0.75]]))
    if got_14 is not None:
        same("18.14", "sa loss (log 2)", got_14[0], np.log(2.0))
        same("18.14", "son gradient", got_14[1], [[0.25, -0.25]])
with checked("18.14"):
    B = mylearn.nn.backward
    rng_14 = np.random.default_rng(1814)
    z_14, t_14 = rng_14.normal(size=(5, 1)) * 3, rng_14.random((5, 1))
    got_14 = pair("18.14", "BCE sur 5 logits", B.binary_cross_entropy_with_logits(z_14, t_14))
    if got_14 is not None:
        same("18.14", "loss BCE, comme -mean(t log σ(z) + (1 - t) log(1 - σ(z)))", got_14[0],
             -np.mean(t_14 * special.log_expit(z_14) + (1 - t_14) * special.log_expit(-z_14)))
        same("18.14", "gradient BCE, comme (σ(z) - t) / 5", got_14[1], (special.expit(z_14) - t_14) / 5)
    with warnings.catch_warnings(record=True) as caught_14, np.errstate(over="warn", invalid="warn", divide="warn", under="ignore"):
        warnings.simplefilter("always")
        extreme_14 = pair("18.14", "BCE de logits ±1000", B.binary_cross_entropy_with_logits([1000.0, -1000.0], [0.0, 0.0]))
    if extreme_14 is not None:
        verdict("18.14", np.isfinite(extreme_14[0]) and not caught_14 and abs(extreme_14[0] - 500.0) < 1e-9,
                "BCE de logits ±1000 : 500, finie et sans avertissement.",
                f"BCE de logits ±1000 : {extreme_14[0]!r}" + (f", avertissement « {caught_14[0].message} »" if caught_14 else "")
                + " (attendu 500 : la forme stable de la fiche n'exponentie que -|z|).")
    logits_14, labels_14 = rng_14.normal(size=(6, 4)) * 2, np.array([0, 3, 1, 1, 2, 0])
    got_14 = pair("18.14", "softmax_cross_entropy avec des labels", B.softmax_cross_entropy(logits_14, labels_14))
    if got_14 is not None:
        logp_14 = special.log_softmax(logits_14, axis=1)
        onehot_14 = np.eye(4)[labels_14]
        same("18.14", "loss avec labels, comme -mean(log p[label])", got_14[0], -logp_14[np.arange(6), labels_14].mean())
        same("18.14", "gradient avec labels, comme (P - Y) / 6", got_14[1], (np.exp(logp_14) - onehot_14) / 6)
        again_14 = pair("18.14", "softmax_cross_entropy avec les mêmes labels en one-hot", B.softmax_cross_entropy(logits_14, onehot_14))
        if again_14 is not None:
            same("18.14", "loss avec la cible one-hot (la même)", again_14[0], got_14[0])
    big_14 = pair("18.14", "softmax_cross_entropy de logits 1e4", B.softmax_cross_entropy([[1e4, 0.0, -1e4]], [2]))
    if big_14 is not None:
        verdict("18.14", np.isfinite(big_14[0]) and abs(big_14[0] - 2e4) < 1e-6, "logits de 1e4 : loss 2e4, finie.",
                f"logits de 1e4 : loss {big_14[0]!r}, attendu 2e4 (calcule log p avec log_softmax, jamais log(softmax)).")
    refuses("18.14", "un label égal à -1", lambda: B.softmax_cross_entropy(np.zeros((2, 3)), np.array([0, -1])))
    refuses("18.14", "une cible (1, 3) pour des logits (2, 3)", lambda: B.softmax_cross_entropy(np.zeros((2, 3)), np.full((1, 3), 1 / 3)))
    refuses("18.14", "une cible BCE égale à 1,5", lambda: B.binary_cross_entropy_with_logits([0.0], [1.5]))
if written(lambda: mylearn.nn.backward.binary_cross_entropy_with_logits([0.0], [1.0])):
    run_mylearn_tests("backward", keyword="test_binary_cross_entropy or test_softmax_cross_entropy")
else:
    print("⏳ Ex 18.14 : les tests attendent tes fonctions.")

### Ex 18.15 — dense_backward : les trois gradients d'une couche 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire la passe arrière d'une couche dense pour un mini-batch, et la vérifier contre des différences finies.  
**Prérequis :** Ex 18.12 · ch. 16 (`dense_forward`, formes) · fiche §18.7, §18.8, §18.12.1, encadré sur les gradients matriciels · **Fil rouge :** synth · **mylearn :** `nn/backward.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/backward.py` (créé par `python tools/start_chapter.py 18`), lis la docstring de la fonction, remplace le `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement ; PyTorch et SciPy sont les **oracles** des tests (sans PyTorch, les tests qui en ont besoin sont ignorés). La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à des calculs indépendants, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch18_backward.py -q` les lance tous dans un terminal.

Écris `dense_backward(dout, x, W)`, la passe arrière de la couche $\mathbf{z} = \mathbf{x}\mathbf{W} + \mathbf{b}$ : elle reçoit `dout` $= \partial L / \partial \mathbf{z}$, de forme $(n, n_{out})$, l'entrée `x` de forme $(n, n_{in})$ et les poids `W` de forme $(n_{in}, n_{out})$, et rend le tuple `(dx, dW, db)` des gradients de la loss par rapport à l'entrée, aux poids et au biais. **Contrat** (la docstring, plus ces précisions que vérifient les tests) : trois **nouveaux** tableaux `float64`, de formes $(n, n_{in})$, $(n_{in}, n_{out})$ et $(n_{out},)$ ; le biais n'est pas un argument (son gradient n'en dépend pas) ; des tableaux qui ne sont pas à deux dimensions ou dont les formes ne s'accordent pas lèvent `ValueError` ; aucune entrée n'est modifiée. Pas de $\frac{1}{n}$ ici : la moyenne du batch est déjà dans `dout`, rendu par la loss.

La vérification essaie l'exemple de la docstring, compare tes trois gradients à des différences finies centrées d'une loss $L = \sum \mathbf{G} \odot \mathbf{z}$ (dont le gradient en $\mathbf{z}$ vaut exactement $\mathbf{G}$), essaie des formes incompatibles, puis lance les tests (`-k test_dense_backward_`).

Dans tes notes : pour $n = 5$, $n_{in} = 3$ et $n_{out} = 4$, écris les formes des six tableaux en jeu, et montre qu'il n'y a qu'une seule façon de les multiplier pour obtenir `dW` et `dx`.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")   # reload your saved file
def triple_15(ex_id, result):
    """Unpack (dx, dW, db), or None with a ❌ line."""
    if result is None:
        print(f"❌ Ex {ex_id} : dense_backward renvoie None : as-tu oublié le return ?")
        return None
    if not isinstance(result, tuple) or len(result) != 3:
        print(f"❌ Ex {ex_id} : dense_backward doit rendre un tuple (dx, dW, db), reçu {type(result).__name__}.")
        return None
    return result


with checked("18.15"):
    B = mylearn.nn.backward
    got_15 = triple_15("18.15", B.dense_backward(np.array([[1.0, -1.0]]), np.array([[1.0, 2.0]]),
                                                  np.array([[1.0, 0.0], [0.0, 1.0]])))
    if got_15 is not None:
        same("18.15", "dx de l'exemple de la docstring", got_15[0], [[1.0, -1.0]])
        same("18.15", "dW de l'exemple de la docstring", got_15[1], [[1.0, -1.0], [2.0, -2.0]])
        same("18.15", "db de l'exemple de la docstring", got_15[2], [1.0, -1.0])
with checked("18.15"):
    B = mylearn.nn.backward
    rng_15 = np.random.default_rng(1815)
    x_15, W_15, b_15 = rng_15.normal(size=(5, 3)), rng_15.normal(size=(3, 4)), rng_15.normal(size=4)
    G_15 = rng_15.normal(size=(5, 4))                 # the gradient that arrives on z

    def loss_15():
        return float(np.sum(G_15 * (x_15 @ W_15 + b_15)))   # a loss whose gradient on z is exactly G_15

    got_15 = triple_15("18.15", B.dense_backward(G_15.copy(), x_15.copy(), W_15.copy()))
    if got_15 is not None:
        same("18.15", "dx contre les différences finies", got_15[0], numeric_grad(loss_15, x_15), rtol=1e-6, atol=1e-8)
        same("18.15", "dW contre les différences finies", got_15[1], numeric_grad(loss_15, W_15), rtol=1e-6, atol=1e-8)
        same("18.15", "db contre les différences finies", got_15[2], numeric_grad(loss_15, b_15), rtol=1e-6, atol=1e-8)
    refuses("18.15", "dout (5, 4) avec x (6, 3)", lambda: B.dense_backward(G_15, np.zeros((6, 3)), W_15))
if written(lambda: mylearn.nn.backward.dense_backward(np.ones((1, 1)), np.ones((1, 1)), np.ones((1, 1)))):
    run_mylearn_tests("backward", keyword="test_dense_backward_")
else:
    print("⏳ Ex 18.15 : les tests attendent ta fonction.")

### Ex 18.16 — gradient_check : vérifier un gradient par différences finies 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire l'outil qui compare un gradient analytique aux différences finies centrées, sans abîmer les paramètres.  
**Prérequis :** Ex 18.15 · ch. 5 (dérivée numérique, choix du pas) · fiche, section sur la vérification d'un gradient · **Fil rouge :** synth · **mylearn :** `nn/backward.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/backward.py` (créé par `python tools/start_chapter.py 18`), lis la docstring de la fonction, remplace le `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement ; PyTorch et SciPy sont les **oracles** des tests (sans PyTorch, les tests qui en ont besoin sont ignorés). La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à des calculs indépendants, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch18_backward.py -q` les lance tous dans un terminal.

Écris `gradient_check(f, param, analytic_grad, eps=1e-6, n_checks=None, rng=None)` : `f` est une fonction **sans argument** qui recalcule la loss en lisant `param` ; `gradient_check` modifie `param` **sur place**, une case à la fois, et rend la plus grande erreur relative $|a - n| / \max(|a| + |n|, 10^{-12})$ entre le gradient analytique $a$ et le gradient numérique $n$ des cases vérifiées. **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- différence **centrée**, avec le pas `eps` donné : $n = \frac{f(w + \varepsilon) - f(w - \varepsilon)}{2\varepsilon}$, une seule case perturbée à la fois ;
- chaque case est remise **bit à bit** à sa valeur d'origine (garde l'ancienne valeur et remets-la : `old + eps - eps` n'est pas toujours `old`), même si `f` lève une erreur ;
- avec `n_checks=None`, toutes les cases ; sinon `n_checks` cases **distinctes** tirées par `np.random.default_rng(rng)` (sans remise), ou toutes s'il y en a moins ;
- `param` doit être un tableau NumPy de **flottants** (sinon `ValueError` : une case entière ne peut pas recevoir $w + 10^{-6}$), et `analytic_grad` doit avoir sa forme (sinon `ValueError`) ;
- la valeur rendue est un `float` Python.

La vérification essaie les exemples de la docstring, vérifie le gradient exact d'une couche dense, puis le même gradient multiplié par 2, contrôle que `param` revient bit à bit, puis lance les tests (`-k test_gradient_check_`).

Dans tes notes : pourquoi un gradient deux fois trop grand donne-t-il **exactement** $\frac13$, quelles que soient les valeurs ? Et un signe faux ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")   # reload your saved file
with checked("18.16"):
    B = mylearn.nn.backward
    w_16 = np.array([1.0, -2.0, 3.0])
    err_16 = B.gradient_check(lambda: float(np.sum(w_16 ** 2)), w_16, 2 * w_16)
    verdict("18.16", isinstance(err_16, float) and err_16 < 1e-7, f"gradient juste de Σw² : erreur relative {err_16:.1e} (< 1e-7).",
            f"gradient juste de Σw² : erreur relative {err_16!r}, attendu un float < 1e-7.")
    err_16 = B.gradient_check(lambda: float(np.sum(w_16 ** 2)), w_16, -2 * w_16)
    verdict("18.16", isinstance(err_16, float) and err_16 > 0.9, f"signe faux : erreur relative {err_16:.3f} (proche de 1).",
            f"signe faux : erreur relative {err_16!r}, attendu un float proche de 1.")
    same("18.16", "w après les deux vérifications (remis à l'identique)", w_16, [1.0, -2.0, 3.0], rtol=0, atol=0)
with checked("18.16"):
    B = mylearn.nn.backward
    rng_16 = np.random.default_rng(1816)
    x_16, W_16, G_16 = rng_16.normal(size=(5, 3)), rng_16.normal(size=(3, 4)), rng_16.normal(size=(5, 4))
    loss_16 = lambda: float(np.sum(G_16 * (x_16 @ W_16)))          # dL/dW = x.T @ G
    exact_16 = x_16.T @ G_16
    saved_16 = W_16.copy()
    err_16 = B.gradient_check(loss_16, W_16, exact_16)
    verdict("18.16", err_16 < 1e-7, f"dW exact d'une couche dense : erreur relative {err_16:.1e}.",
            f"dW exact d'une couche dense : erreur relative {err_16:.3g}, attendu moins de 1e-7.")
    err_16 = B.gradient_check(loss_16, W_16, 2 * exact_16)
    verdict("18.16", abs(err_16 - 1 / 3) < 1e-6, f"facteur 2 en trop : erreur relative {err_16:.4f} (1/3).",
            f"facteur 2 en trop : erreur relative {err_16:.4g}, attendu 1/3 : |2g − g| / (|2g| + |g|).")
    err_16 = B.gradient_check(loss_16, W_16, exact_16, n_checks=3, rng=np.random.default_rng(0))
    verdict("18.16", err_16 < 1e-7, "n_checks=3 avec un générateur : accepté.", f"n_checks=3 : erreur {err_16!r}.")
    verdict("18.16", W_16.tobytes() == saved_16.tobytes(), "W remis bit à bit après toutes les vérifications.",
            "W n'est pas remis bit à bit : garde l'ancienne valeur et remets-la (jamais old + eps − eps).")
    refuses("18.16", "un param d'entiers", lambda: B.gradient_check(lambda: 0.0, np.array([1, 2]), np.array([0.0, 0.0])))
if written(lambda: mylearn.nn.backward.gradient_check(lambda: 0.0, np.zeros(1), np.zeros(1))):
    run_mylearn_tests("backward", keyword="test_gradient_check_")
else:
    print("⏳ Ex 18.16 : les tests attendent ta fonction.")

### Ex 18.17 — Chasse au bug : trois gradients faux démasqués 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver, avec ton gradient_check, trois erreurs dans la passe arrière d'un petit réseau qui tourne sans planter.  
**Prérequis :** Ex 18.16 · fiche §18.10, §18.12.5, section sur la vérification d'un gradient · **Parcours :** C

Un collègue a écrit `grads_17`, la passe arrière d'un petit réseau (4 entrées, 5 neurones cachés tanh, 3 classes, cross-entropy softmax **moyenne** sur 6 exemples). Elle tourne, rend des tableaux de la bonne forme, et l'entraînement « marche à peu près » : il y a pourtant **trois** erreurs.

a) Avec **ton** `gradient_check` (Ex 18.16), mesure l'erreur relative de son `dW2` (la fonction d'origine) : perturbe `W2` d'une copie de `PARAMS_17`, et recalcule `loss_17`. Range la valeur, sans l'arrondir, dans `err_17`.

b) Écris `grads_fixed_17`, la version corrigée (copie le code, puis corrige-le ; même signature, mêmes formes). Ton `gradient_check` sur chacun des quatre gradients est le meilleur guide.

La vérification contrôle trois points, dans l'ordre où les erreurs se masquent (un point ne se teste que si les précédents passent) : 1) `dW2` et `db2` ; 2) `dW1` ; 3) `db1`, chacun contre des différences finies calculées indépendamment de ton code. Cet exercice se corrige aussi avec `05_solutions.md`.

Dans tes notes : pour chaque erreur, l'erreur relative qu'elle produit et pourquoi elle a cette valeur ; laquelle de ces erreurs l'entraînement aurait-il le plus de mal à « absorber » ?

In [ ]:
rng_17 = np.random.default_rng(1817)
X_17 = rng_17.normal(size=(6, 4))                    # 6 examples, 4 features
LABELS_17 = np.array([0, 2, 1, 1, 0, 2])             # 3 classes
PARAMS_17 = [(rng_17.normal(0, 0.5, size=(4, 5)), rng_17.normal(0, 0.1, size=5)),   # hidden layer, tanh
             (rng_17.normal(0, 0.5, size=(5, 3)), rng_17.normal(0, 0.1, size=3))]   # output layer, logits


def loss_17(params):
    """Mean softmax cross-entropy of the network tanh(x W1 + b1) W2 + b2 on (X_17, LABELS_17)."""
    (W1, b1), (W2, b2) = params
    logits = np.tanh(X_17 @ W1 + b1) @ W2 + b2
    return float(-special.log_softmax(logits, axis=1)[np.arange(len(X_17)), LABELS_17].mean())


def grads_17(params):
    """A colleague's backward pass for loss_17: returns [(dW1, db1), (dW2, db2)]. It runs, but some gradients are wrong."""
    (W1, b1), (W2, b2) = params
    n = len(X_17)
    z1 = X_17 @ W1 + b1
    a1 = np.tanh(z1)
    logits = a1 @ W2 + b2
    P = special.softmax(logits, axis=1)
    Y = np.eye(3)[LABELS_17]
    delta2 = P - Y
    dW2, db2 = a1.T @ delta2, delta2.sum(axis=0)
    delta1 = (delta2 @ W2.T) * (1 - np.tanh(a1) ** 2)
    dW1, db1 = X_17.T @ delta1, delta1.mean(axis=0)
    return [(dW1, db1), (dW2, db2)]

In [ ]:
# a) With YOUR gradient_check (Ex 18.16), the relative error of the colleague's dW2 (the original grads_17):
#    perturb W2 of a copy of PARAMS_17 and recompute loss_17; do not round the value.
err_17 = ...


def grads_fixed_17(params):
    """The corrected backward pass: copy the code of grads_17 here, then fix it (same signature, same result shape)."""
    raise NotImplementedError("grads_fixed_17")

In [ ]:
def oracle_17(params):
    """The exact gradients of loss_17, by central differences (this check does not use your gradient_check)."""
    copy = [(W.copy(), b.copy()) for W, b in params]
    return [tuple(numeric_grad(lambda: loss_17(copy), array) for array in layer) for layer in copy]


EXACT_17 = oracle_17(PARAMS_17)
with wb.attempt("18.17"):
    wb.check("18.17a", err_17)
with checked("18.17"):
    got_17 = grads_fixed_17([(W.copy(), b.copy()) for W, b in PARAMS_17])
    shapes_17 = [[np.shape(g) for g in layer] for layer in got_17] if isinstance(got_17, (list, tuple)) else None
    if shapes_17 != [[(4, 5), (5,)], [(5, 3), (3,)]]:
        print(f"❌ Ex 18.17 : grads_fixed_17 doit rendre [(dW1, db1), (dW2, db2)] de formes [(4, 5), (5,)], [(5, 3), (3,)] ; "
              f"reçu {shapes_17 if shapes_17 is not None else type(got_17).__name__}.")
    else:
        points_17 = [("1) dW2 et db2 (la couche de sortie)", [got_17[1][0], got_17[1][1]], [EXACT_17[1][0], EXACT_17[1][1]]),
                     ("2) dW1 (la couche cachée)", [got_17[0][0]], [EXACT_17[0][0]]),
                     ("3) db1 (les biais cachés)", [got_17[0][1]], [EXACT_17[0][1]])]
        for k_17, (label_17, mine_17, exact_17) in enumerate(points_17):
            worst_17 = max(relative_error(m, e) for m, e in zip(mine_17, exact_17))
            if worst_17 >= 1e-6:
                print(f"❌ Ex 18.17 : point {label_17} : erreur relative {worst_17:.3g} contre les différences finies. "
                      "Mesure-la avec ton gradient_check, puis cherche la ligne qui la produit.")
                for later_17 in points_17[k_17 + 1:]:
                    print(f"⏳ Ex 18.17 : point {later_17[0]} : corrige d'abord le point {label_17[:2]}")
                break
            print(f"✅ Ex 18.17 : point {label_17} : juste (erreur relative {worst_17:.1e}).")

## Partie B · Rétropropager tout un réseau, puis l'entraîner

Écrire la rétropropagation de tout un MLP (18.18), puis entraîner le réseau 2-4-4-1 du livre sur les lunes avec trois learning rates (18.19), refaire l'expérience avec scikit-learn (18.20) et tracer ce que le livre montre dans ses figures 18.49 à 18.57 (18.21). La cellule suivante crée les 1 500 lunes standardisées et fixe le nombre d'epochs.

In [ ]:
X_MOONS, y_MOONS = moons_18()                     # the 1 500 standardized moons of the book's experiment
EPOCHS_19 = 40
print("moons:", X_MOONS.shape, "· classes:", np.bincount(y_MOONS).tolist())

### Ex 18.18 — mlp_backward : rétropropager à travers tout le réseau 🔨 ★★★ ⏱️ 45 min
**Objectif :** écrire la passe arrière d'un MLP entier à partir du cache de sa passe avant.  
**Prérequis :** Ex 18.15 · ch. 16 (`mlp_forward` et son cache) · ch. 17 (dérivées en $z$) · fiche §18.8 à §18.10, §18.12.1 · **Fil rouge :** synth · **mylearn :** `nn/backward.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/backward.py` (créé par `python tools/start_chapter.py 18`), lis la docstring de la fonction, remplace le `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement ; PyTorch et SciPy sont les **oracles** des tests (sans PyTorch, les tests qui en ont besoin sont ignorés). La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à des calculs indépendants, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch18_backward.py -q` les lance tous dans un terminal.

Écris `mlp_backward(dout, cache, params, activation_derivative=None, output_activation_derivative=None, return_input_grad=False)`. Elle reçoit `dout` $= \partial L / \partial$ (sortie du réseau), de forme $(n, n_L)$ (le gradient rendu par une loss calculée sur les logits), le `cache` de ton `mlp_forward(..., return_cache=True)` du ch. 16 (un couple `(a_prev, z)` par couche) et la liste `params` des `(W, b)` de la passe avant. **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- elle rend la liste des `(dW, db)`, dans l'ordre et avec les formes de `params` ; avec `return_input_grad=True`, le tuple `(grads, dx)`, `dx` ayant la forme de l'entrée du réseau ;
- les dérivées sont celles du ch. 17, qui prennent la **pré-activation** : `activation_derivative` est appelée **une fois par couche cachée**, sur le `z` du cache ; `output_activation_derivative` s'applique au `z` de la dernière couche (`None` : sortie linéaire, le cas d'une loss sur les logits) ; `None` pour l'une ou l'autre veut dire l'identité ;
- `dout`, le cache et les paramètres ne sont jamais modifiés (calcule `delta = grad * f'(z)` dans un nouveau tableau, jamais avec `*=`) ;
- un cache et des paramètres de longueurs différentes, ou un `dout` de la mauvaise forme, lèvent `ValueError`.

La vérification essaie l'exemple de la docstring, compare tous les gradients d'un réseau 3-5-4-2 (tanh cachées, sortie sigmoïde) à des différences finies, compte les appels de la dérivée cachée, puis lance les tests (`-k test_mlp_backward_`).

Dans tes notes : pourquoi la dérivée doit-elle être évaluée en $z$ (gardé par le cache) et non en $a$ ? Que changerait un `*=` sur `dout` pour l'appelant ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")   # reload your saved file
def grads_of_18(ex_id, result, params, with_dx=False):
    """Unpack the result of mlp_backward (a list of (dW, db), or (grads, dx)), or None with a ❌ line."""
    if result is None:
        print(f"❌ Ex {ex_id} : mlp_backward renvoie None : as-tu oublié le return ?")
        return None
    if with_dx:
        if not isinstance(result, tuple) or len(result) != 2:
            print(f"❌ Ex {ex_id} : avec return_input_grad=True, mlp_backward doit rendre le tuple (grads, dx).")
            return None
        result = result[0]
    if not isinstance(result, list) or len(result) != len(params) or not all(isinstance(g, tuple) and len(g) == 2 for g in result):
        print(f"❌ Ex {ex_id} : mlp_backward doit rendre une liste de {len(params)} tuples (dW, db), reçu {type(result).__name__}.")
        return None
    return result


with checked("18.18"):
    B = mylearn.nn.backward
    x_18 = np.array([[1.0, 1.0]])
    params_18 = [(np.array([[1.0, -1.0], [0.5, -2.0]]), np.array([0.0, 1.0])), (np.array([[2.0], [1.0]]), np.array([0.5]))]
    cache_18 = [(x_18, np.array([[1.5, -2.0]])), (np.array([[1.5, 0.0]]), np.array([[3.5]]))]
    relu_derivative_18 = lambda z: (z > 0).astype(float)
    result_18 = B.mlp_backward(np.array([[1.0]]), cache_18, params_18, activation_derivative=relu_derivative_18,
                               return_input_grad=True)
    grads_18 = grads_of_18("18.18", result_18, params_18, with_dx=True)
    if grads_18 is not None:
        same("18.18", "dW de la couche 0 (exemple de la docstring)", grads_18[0][0], [[2.0, 0.0], [2.0, 0.0]])
        same("18.18", "dW de la couche 1", grads_18[1][0], [[1.5], [0.0]])
        same("18.18", "dx", result_18[1], [[2.0, 1.0]])
with checked("18.18"):
    B = mylearn.nn.backward
    rng_18 = np.random.default_rng(1818)
    sizes_18 = [3, 5, 4, 2]
    params_18 = [(rng_18.normal(0, 0.7, (a, b)), rng_18.normal(0, 0.3, b)) for a, b in zip(sizes_18[:-1], sizes_18[1:])]
    x_18, G_18 = rng_18.normal(size=(6, 3)), rng_18.normal(size=(6, 2))

    def forward_18(params, x):
        """Our own forward pass: tanh hidden layers, sigmoid output; returns (output, cache in the mlp_forward format)."""
        a, cache = x, []
        for layer, (W, b) in enumerate(params):
            z = a @ W + b
            cache.append((a, z))
            a = np.tanh(z) if layer < len(params) - 1 else special.expit(z)
        return a, cache

    def loss_18():
        return float(np.sum(G_18 * forward_18(params_18, x_18)[0]))     # dL/d(output) = G_18

    _, cache_18 = forward_18(params_18, x_18)
    calls_18 = []

    def tanh_derivative_18(z):
        calls_18.append(z)
        return 1 - np.tanh(z) ** 2

    sigmoid_derivative_18 = lambda z: special.expit(z) * (1 - special.expit(z))
    result_18 = B.mlp_backward(G_18.copy(), cache_18, params_18, activation_derivative=tanh_derivative_18,
                               output_activation_derivative=sigmoid_derivative_18, return_input_grad=True)
    grads_18 = grads_of_18("18.18", result_18, params_18, with_dx=True)
    if grads_18 is not None:
        for layer_18, ((W_18, b_18), (dW_18, db_18)) in enumerate(zip(params_18, grads_18)):
            same("18.18", f"dW de la couche {layer_18} (tanh cachées, sortie sigmoïde) contre les différences finies", dW_18,
                 numeric_grad(loss_18, W_18), rtol=1e-6, atol=1e-8)
            same("18.18", f"db de la couche {layer_18} contre les différences finies", db_18, numeric_grad(loss_18, b_18),
                 rtol=1e-6, atol=1e-8)
        same("18.18", "dx contre les différences finies", result_18[1], numeric_grad(loss_18, x_18), rtol=1e-6, atol=1e-8)
        hidden_z_18 = [z for _, z in cache_18[:-1]]
        verdict("18.18", len(calls_18) == 2 and all(any(c is z or np.array_equal(c, z) for c in calls_18) for z in hidden_z_18),
                "la dérivée cachée est appelée une fois par couche cachée, sur le z du cache.",
                f"la dérivée cachée a été appelée {len(calls_18)} fois : attendu une fois par couche cachée (2), sur le z du cache.")
if written(lambda: mylearn.nn.backward.mlp_backward(np.ones((1, 1)), [(np.ones((1, 1)), np.ones((1, 1)))], [(np.ones((1, 1)), np.ones(1))])):
    run_mylearn_tests("backward", keyword="test_mlp_backward_")
else:
    print("⏳ Ex 18.18 : les tests attendent ta fonction.")

### Ex 18.19 — Entraîner le réseau 2-4-4-1 à 37 poids sur les lunes 🔨 ★★★ ⏱️ 45 min
**Objectif :** écrire la boucle d'entraînement du livre avec ta librairie, et la lancer avec trois learning rates.  
**Prérequis :** Ex 18.18 · Ex 18.14 · ch. 16 (`init_mlp`, `mlp_forward`) · 0A (`iterate_minibatches`) · fiche §18.9, §18.11.1, §18.12.5 · **Fil rouge :** synth · **Parcours :** R, C

Le réseau du livre (§18.11.1) : 2 entrées, deux couches cachées de 4 neurones ReLU, une sortie linéaire, le **logit** ; 28 poids et 9 biais. Écris `train_19(X, y, lr, epochs, seed=0, batch_size=1)`, qui l'entraîne avec **ta** librairie. **Contrat** (le protocole est imposé, pour que tes nombres soient ceux de la vérification) :
- un seul générateur, `rng = np.random.default_rng(seed)`, crée d'abord les poids par `init_mlp([2, 4, 4, 1], "he_normal", rng)` (ch. 16), puis mélange les exemples à chaque epoch par `iterate_minibatches(len(X), batch_size, rng=rng)` (0A) ;
- pour chaque mini-batch : `mlp_forward` avec `relu` (ch. 17), la cross-entropy binaire **sur les logits** (Ex 18.14) contre les labels mis en colonne `(n, 1)`, `mlp_backward` avec `relu_derivative` (Ex 18.18), puis la mise à jour $\mathbf{W} \leftarrow \mathbf{W} - \eta\,d\mathbf{W}$, $\mathbf{b} \leftarrow \mathbf{b} - \eta\,d\mathbf{b}$ de toutes les couches ;
- avec `batch_size=1`, c'est la descente de gradient **stochastique** du livre : une mise à jour par exemple ;
- après chaque epoch, sur **tout** `(X, y)` : la loss moyenne, l'accuracy (logit $> 0$ ⇒ classe 1) et les 5 paramètres de la couche de sortie (ses 4 poids, puis son biais), dans le dictionnaire `history` décrit par la docstring.

La vérification entraîne le réseau 40 epochs avec $\eta = 0{,}5$, $0{,}05$ et $0{,}01$ (une vingtaine de secondes), garde les trois entraînements dans `RUNS_19` pour 18.20 et 18.21, puis vérifie quatre valeurs calculées par ton code : a) l'accuracy finale avec $\eta = 0{,}5$ ; b) la loss après la première epoch avec $\eta = 0{,}01$ ; c) la même avec $\eta = 0{,}05$ ; d) l'accuracy finale avec $\eta = 0{,}05$.

Dans tes notes : compare les trois entraînements à ce que décrit la fiche (§18.11.1). Quels phénomènes retrouves-tu ? Lesquels tes chiffres changent-ils ?

In [ ]:
def train_19(X, y, lr, epochs, seed=0, batch_size=1):
    """Train the 2-4-4-1 network of the book (ReLU hidden layers, a linear output: the logit) by stochastic gradient
    descent on the binary cross-entropy of the logits, with the protocol of the statement.

    Returns (params, history): params = the trained [(W, b), (W, b), (W, b)]; history = a dict of three lists, one
    entry per epoch, measured on all of (X, y) after the epoch: "loss" (floats), "accuracy" (floats) and
    "output_layer" (arrays of shape (5,): the 4 weights of the output layer, then its bias).
    """
    raise NotImplementedError("train_19")

In [ ]:
RUNS_19 = {}
with checked("18.19"):
    for lr_19 in (0.5, 0.05, 0.01):
        started_19 = time.perf_counter()
        result_19 = train_19(X_MOONS, y_MOONS, lr_19, EPOCHS_19, seed=0)
        if not (isinstance(result_19, tuple) and len(result_19) == 2 and isinstance(result_19[1], dict)):
            print(f"❌ Ex 18.19 : train_19 doit rendre le tuple (params, history), history étant un dictionnaire ; reçu {type(result_19).__name__}.")
            break
        history_19 = result_19[1]
        if set(history_19) != {"loss", "accuracy", "output_layer"} or any(len(history_19[k]) != EPOCHS_19 for k in history_19):
            print(f'❌ Ex 18.19 : history doit avoir les clés "loss", "accuracy" et "output_layer", chacune une liste de {EPOCHS_19} valeurs (une par epoch).')
            break
        if np.shape(history_19["output_layer"]) != (EPOCHS_19, 5):
            print("❌ Ex 18.19 : chaque entrée de history[\"output_layer\"] doit être un tableau de 5 nombres (4 poids, puis le biais).")
            break
        RUNS_19[lr_19] = result_19
        acc_19 = history_19["accuracy"]
        print(f"η = {lr_19:<5} accuracy after epochs 1, 10, 20, 40: {[round(acc_19[k], 4) for k in (0, 9, 19, 39)]} · "
              f"final loss {history_19['loss'][-1]:.4f} · {time.perf_counter() - started_19:.1f} s")
if len(RUNS_19) == 3:
    wb.check("18.19a", RUNS_19[0.5][1]["accuracy"][-1], computed=True)
    wb.check("18.19b", RUNS_19[0.01][1]["loss"][0], computed=True)
    wb.check("18.19c", RUNS_19[0.05][1]["loss"][0], computed=True)
    wb.check("18.19d", RUNS_19[0.05][1]["accuracy"][-1], computed=True)

### Ex 18.20 — La même expérience avec MLPClassifier(solver='sgd') 📦 ★★ ⏱️ 20 min
**Objectif :** retrouver avec scikit-learn les trois régimes du learning rate, en réglant `MLPClassifier` comme ta boucle.  
**Prérequis :** Ex 18.19 · ch. 15 (`MLPClassifier`) · fiche §18.11.1 · **Fil rouge :** synth · **Parcours :** C

Écris `sk_20(lr, seed=0, epochs=EPOCHS_20)`, qui rend un **nouveau** `MLPClassifier` entraîné sur `(X_MOONS, y_MOONS)` et réglé aussi près que possible du protocole de 18.19 : deux couches cachées de 4 neurones ReLU, la descente de gradient stochastique **sans** momentum (le défaut de scikit-learn en a un, de 0,9), **sans** pénalité L2 (`alpha`, $10^{-4}$ par défaut), une mise à jour par exemple, `epochs` passages sur les données mélangés à chaque fois, et **sans** arrêt anticipé (scikit-learn s'arrête quand la loss ne baisse plus assez : règle `tol` et `n_iter_no_change` pour qu'il fasse toutes ses epochs). Lis la documentation de `MLPClassifier` pour trouver les noms des paramètres.

La vérification entraîne trois réseaux ($\eta = 0{,}5$, $0{,}05$, $0{,}01$ ; une trentaine de secondes : scikit-learn est lent avec une mise à jour par exemple), contrôle les réglages, compare les accuracies à celles de ton `train_19` et trace les deux courbes de loss.

Dans tes notes : les deux courbes de loss ne coïncident pas, même quand les deux réseaux apprennent. Cite deux différences entre les deux protocoles qui l'expliquent (regarde comment scikit-learn initialise les poids et ce que mesure `loss_curve_`).

In [ ]:
EPOCHS_20 = 20 if FAST_MODE else EPOCHS_19

In [ ]:
def sk_20(lr, seed=0, epochs=EPOCHS_20):
    """A NEW MLPClassifier, fitted on (X_MOONS, y_MOONS), that reproduces the protocol of 18.19 as closely as
    scikit-learn allows (see the statement)."""
    raise NotImplementedError("sk_20")

In [ ]:
SK_20 = {}
with checked("18.20"):
    for lr_20 in (0.5, 0.05, 0.01):
        model_20 = sk_20(lr_20)
        if not isinstance(model_20, MLPClassifier) or not hasattr(model_20, "loss_curve_"):
            print(f"❌ Ex 18.20 : sk_20 doit rendre un MLPClassifier entraîné, reçu {type(model_20).__name__}.")
            break
        p_20 = model_20.get_params()
        wrong_20 = {k: p_20[k] for k, v in {"solver": "sgd", "momentum": 0.0, "alpha": 0.0, "batch_size": 1,
                                             "hidden_layer_sizes": (4, 4), "activation": "relu"}.items()
                    if (tuple(p_20[k]) if k == "hidden_layer_sizes" else p_20[k]) != v}
        if wrong_20:
            print(f"❌ Ex 18.20 : réglages différents du protocole de 18.19 : {wrong_20} (relis l'énoncé).")
            break
        if model_20.n_iter_ != EPOCHS_20:
            print(f"❌ Ex 18.20 : {model_20.n_iter_} epochs au lieu de {EPOCHS_20} : l'arrêt anticipé de scikit-learn s'est déclenché "
                  "(tol, n_iter_no_change).")
            break
        SK_20[lr_20] = model_20
        print(f"η = {lr_20:<5} scikit-learn accuracy {model_20.score(X_MOONS, y_MOONS):.4f}" +
              (f" · your train_19 (same epochs) {RUNS_19[lr_20][1]['accuracy'][EPOCHS_20 - 1]:.4f}" if lr_20 in RUNS_19 else ""))
if len(SK_20) == 3:
    fig_20, axes_20 = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
    for ax, (lr_20, model_20) in zip(axes_20, SK_20.items()):
        ax.plot(range(1, EPOCHS_20 + 1), model_20.loss_curve_, label="scikit-learn (loss_curve_)")
        if lr_20 in RUNS_19:
            ax.plot(range(1, EPOCHS_20 + 1), RUNS_19[lr_20][1]["loss"][:EPOCHS_20], "--", label="ton train_19")
        ax.set_title(f"η = {lr_20}")
        ax.set_xlabel("epoch")
    axes_20[0].set_ylabel("loss")
    axes_20[0].legend()
    plt.tight_layout()
    plt.show()
    verdict("18.20", SK_20[0.5].score(X_MOONS, y_MOONS) <= 0.55 < SK_20[0.05].score(X_MOONS, y_MOONS),
            "avec scikit-learn aussi, η = 0,5 bloque le réseau sur une seule classe, et η = 0,05 apprend.",
            "les accuracies de scikit-learn sont inattendues : vérifie les réglages (momentum, alpha, batch_size).")

### Ex 18.21 — Reproduire les figures 18.49 à 18.57 (η = 0,5 ; 0,05 ; 0,01) 🎨 ★★ ⏱️ 30 min
**Objectif :** tracer, pour chaque learning rate, les courbes d'apprentissage, la frontière et les poids de sortie, puis y lire trois faits.  
**Prérequis :** Ex 18.19 · ch. 15 (`plot_decision_boundary`) · fiche §18.11.1 · **Fil rouge :** synth · **Parcours :** C

Avec les trois entraînements de `RUNS_19` (Ex 18.19 ; `RUNS_19[lr]` est le couple `(params, history)`), trace une figure de trois lignes (une par learning rate) et trois colonnes : la loss et l'accuracy par epoch, la frontière de décision du réseau entraîné (`wb.plot.plot_decision_boundary(NetModel_21(params), X_MOONS, y_MOONS, ax=..., title=...)`), et les 5 paramètres de la couche de sortie par epoch. Puis calcule trois valeurs dans `RUNS_19` (par du code, pas à l'œil) :
a) `perfect_21` : avec $\eta = 0{,}05$, la première epoch (comptée à partir de 1) où l'accuracy vaut exactement 1 ;
b) `plateau_21` : avec $\eta = 0{,}01$, le nombre d'epochs où l'accuracy est strictement sous 0,9 ;
c) `moving_21` : avec $\eta = 0{,}5$, combien des 5 paramètres de la couche de sortie ont une valeur différente à la 1ʳᵉ et à la 40ᵉ epoch.

Dans tes notes : à $\eta = 0{,}5$, pourquoi certains paramètres de sortie ne bougent-ils plus du tout ? Que fait le biais de sortie, et quelle loss vise-t-il ? À $\eta = 0{,}01$, à quoi ressemble la frontière pendant le plateau ?

In [ ]:
# 1) The figure: one row per learning rate of RUNS_19 (0.5, 0.05, 0.01), three panels per row:
#    the loss and the accuracy per epoch, the decision boundary of the trained network (use NetModel_21 below),
#    the 5 parameters of the output layer per epoch. Finish with plt.show().
class NetModel_21:
    """Wrap trained params so that wb.plot.plot_decision_boundary can call .predict."""
    def __init__(self, params):
        self.params = params

    def predict(self, X):
        return (mylearn.nn.layers.mlp_forward(X, self.params, mylearn.nn.activations.relu)[:, 0] > 0).astype(int)


# your plotting code here, then plt.show()

# 2) Three values read in RUNS_19 (computed with code, not by eye)
perfect_21 = ...    # a) η = 0.05: the first epoch (counted from 1) where the accuracy equals 1
plateau_21 = ...    # b) η = 0.01: the number of epochs where the accuracy is below 0.9
moving_21 = ...     # c) η = 0.5: how many of the 5 output parameters change between epoch 1 and epoch 40

In [ ]:
with wb.attempt("18.21"):
    wb.check("18.21a", perfect_21)
    wb.check("18.21b", plateau_21)
    wb.check("18.21c", moving_21)

## Partie C · Diagnostiquer un entraînement

Reconnaître un learning rate mal réglé ou un bug sur des courbes de loss (18.22), balayer le learning rate du réseau des lunes sur cinq graines (18.23), voir des neurones mourir ou saturer pendant un entraînement sur MNIST (18.24), puis écrire un test pytest qui attrape les gradients faux d'une couche (18.25).

### Ex 18.22 — Lire des courbes de loss : divergence, lenteur, plateau, bug 📈 ★★ ⏱️ 20 min
**Objectif :** reconnaître, sur une courbe de loss, un learning rate trop grand ou trop petit, un plateau et deux bugs.  
**Prérequis :** 18.Q8 · ✏️ 18.3 · fiche §18.11, §18.11.1, §18.12.8 · **Fil rouge :** synth · **Parcours :** R, M

Six entraînements d'un même classifieur à $K$ classes équilibrées (cross-entropy moyenne en nats, tracée à chaque epoch ; les courbes sont fabriquées, mais leurs formes sont celles qu'on rencontre). Chacun souffre d'un des problèmes ci-dessous, sauf un qui se passe bien. Pour chaque question, la **lettre** de la courbe :
a) `too_large_22` : un learning rate beaucoup trop grand ;
b) `too_small_22` : un learning rate beaucoup trop petit ;
c) `plateau_22` : une région presque plate de la surface de la loss, traversée puis quittée ;
d) `sign_22` : une erreur de signe dans la mise à jour, $w \leftarrow w + \eta\,\partial L / \partial w$ ;
e) `frozen_22` : des poids qui ne sont jamais mis à jour (mise à jour oubliée, ou $\eta = 0$) ;
f) `classes_22` : toutes les courbes partent de la loss d'un réseau qui donne la même probabilité $\frac1K$ à chaque classe ; combien de classes le problème compte-t-il ?

Dans tes notes : pour chaque problème, le premier geste que tu ferais (fiche §18.12.8) ; quelle courbe pourrait aussi venir d'un gradient faux, et comment le vérifier ?

In [ ]:
def curves_22():
    """Six synthetic training curves of a 10-class classifier (mean cross-entropy per epoch), labelled A to F."""
    rng = np.random.default_rng(1822)
    t = np.arange(1, 61, dtype=float)
    start = np.log(10.0)
    noise = lambda size=60: 1 + 0.01 * rng.normal(size=size)
    plateau = np.where(t < 6, start - (start - 1.4) * t / 6, 1.4)
    plateau = np.where(t > 32, 1.4 - 1.25 * (1 - np.exp(-(t - 32) / 4)), plateau) * noise()
    explode = start + 0.12 * 1.4 ** t * (1 + np.sin(2.4 * t))
    explode[t > 11] = np.nan                                  # the loss became inf, then nan
    return {"A": plateau,
            "B": (0.08 + (start - 0.08) * np.exp(-t / 6)) * noise(),
            "C": (start + 0.05 * t + 0.0004 * t ** 2) * noise(),
            "D": (start - 0.007 * t) * noise(),
            "E": explode,
            "F": np.full(60, start)}


CURVES_22 = curves_22()
fig_22, axes_22 = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, (name, curve) in zip(axes_22.ravel(), CURVES_22.items()):
    ax.plot(np.arange(1, 61), curve, color="#2a78d6")
    ax.set_title(f"courbe {name}")
    ax.set_ylim(0, 6)
    ax.grid(alpha=0.3)
for ax in axes_22[1]:
    ax.set_xlabel("epoch")
for ax in axes_22[:, 0]:
    ax.set_ylabel("loss d'entraînement")
fig_22.suptitle("Six entraînements d'un classifieur à K classes équilibrées (cross-entropy moyenne, en nats)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
too_large_22 = ...   # a) the letter of the curve
too_small_22 = ...   # b)
plateau_22 = ...     # c)
sign_22 = ...        # d)
frozen_22 = ...      # e)
classes_22 = ...     # f) the number of classes K (a whole number)

In [ ]:
with wb.attempt("18.22"):
    wb.check("18.22a", too_large_22)
    wb.check("18.22b", too_small_22)
    wb.check("18.22c", plateau_22)
    wb.check("18.22d", sign_22)
    wb.check("18.22e", frozen_22)
    wb.check("18.22f", classes_22)

### Ex 18.23 — Balayage du learning rate sur cinq graines 🔬 ★★★ ⏱️ 40 min
**Objectif :** balayer le learning rate du réseau des lunes sur une grille logarithmique et cinq graines, et lire la plage utile.  
**Prérequis :** Ex 18.19 · fiche §18.11 (encadré sur le choix du learning rate), §18.12.8 · **Fil rouge :** synth · **Parcours :** C

Écris `sweep_23(lrs, seeds, epochs)`, qui entraîne le réseau de 18.19 (avec **ton** `train_19`, `batch_size=1`) pour chaque learning rate de `lrs` et chaque graine de `seeds`, et rend deux tableaux de forme `(len(lrs), len(seeds))` : la loss et l'accuracy sur `(X_MOONS, y_MOONS)` après la dernière epoch. La vérification lance 25 entraînements de 10 epochs (une minute environ), affiche le tableau, puis trace la loss médiane avec sa dispersion et l'accuracy de chaque graine, en fonction de $\eta$ (échelle logarithmique).

Puis réponds, d'après le tableau affiché :
a) `best_23` : la valeur de `LRS_23` qui donne la plus petite loss **médiane** ;
b) `stuck_23` : avec $\eta = 0{,}3$, combien des cinq graines finissent avec une accuracy d'au plus 0,55 (un réseau qui répond presque toujours la même classe) ?

Dans tes notes : pourquoi regarder la médiane et la dispersion plutôt qu'une seule graine ? La plus petite loss de **toutes** les graines est-elle obtenue au même $\eta$ que la plus petite médiane ? Quel $\eta$ choisirais-tu pour un nouveau réseau, et pourquoi pas le plus grand qui marche parfois ?

In [ ]:
LRS_23 = [0.003, 0.01, 0.03, 0.1, 0.3]
SEEDS_23 = [0, 1, 2, 3, 4]
EPOCHS_23 = 10

In [ ]:
def sweep_23(lrs, seeds, epochs):
    """Train the network of 18.19 (YOUR train_19, batch_size=1) for every learning rate and every seed.

    Returns (losses, accuracies): two arrays of shape (len(lrs), len(seeds)), the loss and the accuracy on
    (X_MOONS, y_MOONS) after the last epoch.
    """
    raise NotImplementedError("sweep_23")

In [ ]:
SWEEP_23 = None
with checked("18.23"):
    started_23 = time.perf_counter()
    result_23 = sweep_23(LRS_23, SEEDS_23, EPOCHS_23)
    shape_23 = (len(LRS_23), len(SEEDS_23))
    if not (isinstance(result_23, tuple) and len(result_23) == 2 and all(np.shape(r) == shape_23 for r in result_23)):
        print(f"❌ Ex 18.23 : sweep_23 doit rendre le tuple (losses, accuracies) de deux tableaux de forme {shape_23}.")
    else:
        SWEEP_23 = tuple(np.asarray(r, dtype=float) for r in result_23)
        print(f"{len(LRS_23) * len(SEEDS_23)} entraînements en {time.perf_counter() - started_23:.0f} s")
if SWEEP_23 is not None:
    losses_23, accuracies_23 = SWEEP_23
    for lr_23, l_23, a_23 in zip(LRS_23, losses_23, accuracies_23):
        print(f"η = {lr_23:<6} losses {np.round(l_23, 3).tolist()} · median {np.median(l_23):.3f} · accuracies {np.round(a_23, 3).tolist()}")
    fig_23, (ax_l, ax_a) = plt.subplots(1, 2, figsize=(12, 3.8))
    ax_l.plot(LRS_23, np.median(losses_23, axis=1), "o-", label="médiane des 5 graines")
    ax_l.fill_between(LRS_23, losses_23.min(axis=1), losses_23.max(axis=1), alpha=0.25, label="de la meilleure à la pire")
    ax_l.set_xscale("log")
    ax_l.set_yscale("log")
    ax_l.set_xlabel("learning rate η (échelle log)")
    ax_l.set_ylabel(f"loss après {EPOCHS_23} epochs")
    ax_l.legend()
    for j_23, seed_23 in enumerate(SEEDS_23):
        ax_a.plot(LRS_23, accuracies_23[:, j_23], "o-", alpha=0.7, label=f"graine {seed_23}")
    ax_a.set_xscale("log")
    ax_a.set_xlabel("learning rate η (échelle log)")
    ax_a.set_ylabel("accuracy")
    ax_a.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

In [ ]:
best_23 = ...    # a) the learning rate of LRS_23 with the smallest MEDIAN loss (read in the table above)
stuck_23 = ...   # b) with η = 0.3, how many of the 5 seeds end with an accuracy of at most 0.55

In [ ]:
with wb.attempt("18.23"):
    wb.check("18.23a", best_23)
    wb.check("18.23b", stuck_23)

### Ex 18.24 — Neurones saturés et ReLU mortes pendant l'entraînement 🔬 ★★★ ⏱️ 45 min
**Objectif :** mesurer, pendant un entraînement sur MNIST, la part de neurones ReLU morts et de valeurs sigmoïdes saturées.  
**Prérequis :** Ex 18.19 · ch. 16 (initialisations) · ch. 17 (saturation) · fiche §18.12.4 · **Fil rouge :** MNIST · **Parcours :** C

Sur 10 000 images de MNIST, un MLP 784-100-100-10 est entraîné 5 epochs par `train_24` (fournie, avec **ta** librairie : `init_weights`, `mlp_forward`, `softmax_cross_entropy`, `mlp_backward`), qui garde, avant l'entraînement puis après chaque epoch, les pré-activations des deux couches cachées sur 1 000 images (`PROBE_24`). Trois réglages : ReLU, poids de He et $\eta = 0{,}05$ ; ReLU, poids de He et $\eta = 1$ ; sigmoïde, poids tirés de $\mathcal{N}(0 ;\ 1)$ et $\eta = 0{,}5$.

Écris deux mesures, qui reçoivent un tableau `z` de forme `(n_exemples, n_neurones)` :
- `dead_fraction_24(z)` : la part des **neurones** (les colonnes) dont la pré-activation est $\le 0$ pour **tous** les exemples : les ReLU mortes de ce batch ;
- `saturated_fraction_24(z, threshold=4.6)` : la part des **valeurs** de `z` telles que $|z| > 4{,}6$ (là, $\sigma'(z) < 0{,}01$ : une sigmoïde saturée).

La vérification essaie tes deux mesures sur un petit tableau, lance les trois entraînements (une dizaine de secondes), trace l'accuracy, la part de neurones morts des deux réseaux ReLU et la part de valeurs saturées du réseau sigmoïde, et contrôle deux ordres de grandeur.

Dans tes notes : que devient le réseau ReLU avec $\eta = 1$, et pourquoi ses neurones morts ne reviennent-ils pas ? Le réseau sigmoïde apprend-il malgré la saturation, et à quelle vitesse ? Quel remède de la fiche (§18.12.4) appliquerais-tu à chacun ?

In [ ]:
X_24, Y_24 = wb.datasets.load_mnist("train", n=10_000, flatten=True, normalize=True, seed=1824)
PROBE_24 = X_24[:1000]                              # the images on which the neurons are watched


def train_24(activation, method, lr, epochs=5, scale=0.05, seed=0, batch_size=64):
    """Train a 784-100-100-10 MLP on X_24 with YOUR library (softmax cross-entropy, SGD), and keep, before training
    and after every epoch, the pre-activations of the two hidden layers on PROBE_24 and the accuracy on X_24."""
    A, L, B = mylearn.nn.activations, mylearn.nn.layers, mylearn.nn.backward
    f, df = A.get_activation(activation)
    rng = np.random.default_rng(seed)
    sizes = [784, 100, 100, 10]
    params = [(L.init_weights(a, b, method, rng, scale=scale), np.zeros(b)) for a, b in zip(sizes[:-1], sizes[1:])]
    record = {"z": [], "accuracy": []}
    for epoch in range(epochs + 1):
        if epoch > 0:
            for idx in mylearn.utils.iterate_minibatches(len(X_24), batch_size, rng=rng):
                out, cache = L.mlp_forward(X_24[idx], params, f, None, return_cache=True)
                _, dout = B.softmax_cross_entropy(out, Y_24[idx])
                grads = B.mlp_backward(dout, cache, params, activation_derivative=df)
                params = [(W - lr * dW, b - lr * db) for (W, b), (dW, db) in zip(params, grads)]
        _, cache = L.mlp_forward(PROBE_24, params, f, None, return_cache=True)
        record["z"].append([z for _, z in cache[:-1]])           # two arrays of shape (1000, 100)
        record["accuracy"].append(float(np.mean(L.mlp_forward(X_24, params, f).argmax(axis=1) == Y_24)))
    return record


CONFIGS_24 = {"ReLU, He, η = 0,05": ("relu", "he_normal", 0.05, 0.05),
              "ReLU, He, η = 1": ("relu", "he_normal", 1.0, 0.05),
              "sigmoïde, N(0, 1), η = 0,5": ("sigmoid", "normal", 0.5, 1.0)}
print("MNIST:", X_24.shape, "· probe:", PROBE_24.shape)

In [ ]:
def dead_fraction_24(z):
    """The fraction of the neurons (columns of z, shape (n_samples, n_neurons)) whose pre-activation is <= 0 for
    EVERY sample: the dead ReLU neurons of this batch."""
    raise NotImplementedError("dead_fraction_24")


def saturated_fraction_24(z, threshold=4.6):
    """The fraction of the entries of z where |z| > threshold (for a sigmoid, sigma'(4.6) < 0.01: a saturated value)."""
    raise NotImplementedError("saturated_fraction_24")

In [ ]:
with checked("18.24"):
    small_24 = np.array([[-1.0, 2.0, 0.0, -3.0], [-0.5, -1.0, 0.0, 5.0], [-2.0, 0.5, 0.0, -6.0]])
    verdict("18.24", dead_fraction_24(small_24) == 0.5,
            "dead_fraction_24 : 2 neurons sur 4 jamais actifs (z ≤ 0 partout, 0 compris), soit 0,5.",
            f"dead_fraction_24 sur un petit exemple : {dead_fraction_24(small_24)!r}, attendu 0,5 "
            "(une colonne = un neurone ; mort si z ≤ 0 pour TOUS les exemples).")
    verdict("18.24", saturated_fraction_24(small_24) == 2 / 12,
            "saturated_fraction_24 : 2 valeurs sur 12 au-delà de 4,6 en valeur absolue.",
            f"saturated_fraction_24 sur un petit exemple : {saturated_fraction_24(small_24)!r}, attendu 2/12 (|z| > 4,6, parmi toutes les valeurs).")
RECORDS_24 = {}
if written(lambda: dead_fraction_24(np.zeros((1, 1)))) and written(lambda: saturated_fraction_24(np.zeros((1, 1)))):
    with checked("18.24"):
        started_24 = time.perf_counter()
        for name_24, (act_24, method_24, lr_24, scale_24) in CONFIGS_24.items():
            RECORDS_24[name_24] = train_24(act_24, method_24, lr_24, scale=scale_24)
        print(f"three trainings in {time.perf_counter() - started_24:.0f} s")
if len(RECORDS_24) == 3:
    fig_24, axes_24 = plt.subplots(1, 3, figsize=(14, 3.6))
    for name_24, record_24 in RECORDS_24.items():
        epochs_24 = range(len(record_24["accuracy"]))
        measure_24 = saturated_fraction_24 if "sigmo" in name_24 else dead_fraction_24
        axes_24[0].plot(epochs_24, record_24["accuracy"], "o-", label=name_24)
        for layer_24, ls_24 in ((0, "-"), (1, "--")):
            axes_24[1 if measure_24 is dead_fraction_24 else 2].plot(
                epochs_24, [measure_24(z[layer_24]) for z in record_24["z"]], ls_24, marker="o",
                label=f"{name_24}, couche {layer_24 + 1}")
        print(f"{name_24:28s} accuracy {[round(a, 3) for a in record_24['accuracy']]}")
        for layer_24 in (0, 1):
            print(f"{'':28s} layer {layer_24 + 1}: dead {[round(dead_fraction_24(z[layer_24]), 2) for z in record_24['z']]}"
                  f" · saturated {[round(saturated_fraction_24(z[layer_24]), 2) for z in record_24['z']]}")
    axes_24[0].set_title("accuracy d'entraînement")
    axes_24[1].set_title("part de neurones ReLU morts")
    axes_24[2].set_title("part de valeurs sigmoïdes saturées")
    for ax in axes_24:
        ax.set_xlabel("epoch (0 : avant l'entraînement)")
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
    relu_big_24 = RECORDS_24["ReLU, He, η = 1"]
    verdict("18.24", dead_fraction_24(relu_big_24["z"][-1][0]) > 0.3 > dead_fraction_24(RECORDS_24["ReLU, He, η = 0,05"]["z"][-1][0]),
            "avec η = 1, une grande part des neurones ReLU de la première couche est morte ; avec η = 0,05, presque aucun.",
            "les parts de neurones morts sont inattendues : vérifie ta librairie (18.18, 18.14) et dead_fraction_24.")
    verdict("18.24", saturated_fraction_24(RECORDS_24["sigmoïde, N(0, 1), η = 0,5"]["z"][0][0]) > 0.3,
            "avec des poids N(0, 1), plus d'un tiers des valeurs sigmoïdes de la première couche sont saturées dès le départ.",
            "la part de valeurs saturées au départ est inattendue : relis saturated_fraction_24.")

### Ex 18.25 — Écrire un test pytest de gradient pour une couche 🛠️ ★★ ⏱️ 25 min
**Objectif :** écrire des tests pytest, dont un paramétré, qui comparent la passe arrière d'une couche aux différences finies et attrapent cinq bugs.  
**Prérequis :** Ex 18.16 · ch. 17 (🛠️ 17.22, tests paramétrés) · 0A (`wb.run_pytest`) · fiche, section sur la vérification d'un gradient · **Parcours :** C

La fonction `layer_ok(dout, x, W, b)` (fournie) est la passe arrière, juste, de la couche $\mathbf{a} = \tanh(\mathbf{x}\mathbf{W} + \mathbf{b})$ : elle reçoit `dout` $= \partial L / \partial \mathbf{a}$ et rend `(dx, dW, db)`, sans modifier ses entrées ; `layer_forward(x, W, b)` calcule la passe avant. Écris une **batterie de tests** pytest pour une fonction `layer_backward(dout, x, W, b)` qui a ce contrat : au moins un test **paramétré** (`@pytest.mark.parametrize`, par exemple sur plusieurs formes), et un test qui compare les trois gradients à des **différences finies centrées** d'une loss $L = \sum \mathbf{G} \odot \mathbf{a}$ (dont le gradient en $\mathbf{a}$ vaut exactement $\mathbf{G}$).

Range tes tests dans la liste `my_tests_25`. Comme en 17.22, la vérification lance vraiment pytest : tes tests doivent **passer** sur `layer_ok` et **échouer** sur chacune des cinq versions buggées fournies, `layer_bug_1` à `layer_bug_5` (lis-les : que fait chacune de travers ?). Tes tests ne peuvent utiliser que `layer_backward`, `layer_forward`, `np`, `math` et `pytest` ; s'ils ont besoin d'une fonction à toi (des différences finies), définis-la sous le nom `numeric_25`, la vérification la copie avec eux. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : pour chaque version buggée, le test qui l'attrape. Laquelle passerait un test de différences finies écrit sur une seule forme carrée ?

In [ ]:
import math

import pytest                                     # for the @pytest.mark.parametrize decorators of your tests


def layer_forward(x, W, b):
    """Forward pass of the layer: a = tanh(x @ W + b)."""
    return np.tanh(x @ W + b)


def layer_ok(dout, x, W, b):
    """Backward pass of the layer a = tanh(x @ W + b): given dout = dL/da, returns (dx, dW, db) (correct)."""
    z = x @ W + b
    delta = dout * (1 - np.tanh(z) ** 2)
    return delta @ W.T, x.T @ delta, delta.sum(axis=0)


def layer_bug_1(dout, x, W, b):
    """Backward pass of the layer a = tanh(x @ W + b): given dout = dL/da, returns (dx, dW, db) (with one bug)."""
    z = x @ W + b
    delta = dout * (1 - np.tanh(z) ** 2)
    return delta @ W.T, x.T @ dout, delta.sum(axis=0)


def layer_bug_2(dout, x, W, b):
    """Backward pass of the layer a = tanh(x @ W + b): given dout = dL/da, returns (dx, dW, db) (with one bug)."""
    z = x @ W + b
    delta = dout * (1 - np.tanh(np.tanh(z)) ** 2)
    return delta @ W.T, x.T @ delta, delta.sum(axis=0)


def layer_bug_3(dout, x, W, b):
    """Backward pass of the layer a = tanh(x @ W + b): given dout = dL/da, returns (dx, dW, db) (with one bug)."""
    z = x @ W + b
    delta = dout * (1 - np.tanh(z) ** 2)
    return delta @ W.T, x.T @ delta, delta.mean(axis=0)


def layer_bug_4(dout, x, W, b):
    """Backward pass of the layer a = tanh(x @ W + b): given dout = dL/da, returns (dx, dW, db) (with one bug)."""
    z = x @ W + b
    delta = dout * (1 - np.tanh(z) ** 2)
    return dout @ W.T, x.T @ delta, delta.sum(axis=0)


def layer_bug_5(dout, x, W, b):
    """Backward pass of the layer a = tanh(x @ W + b): given dout = dL/da, returns (dx, dW, db) (with one bug)."""
    z = x @ W + b
    dout *= 1 - np.tanh(z) ** 2         # "saves" a copy
    delta = dout
    return delta @ W.T, x.T @ delta, delta.sum(axis=0)

In [ ]:
# Write your test functions here (their names start with test_). They test a function layer_backward(dout, x, W, b),
# whose contract is the one of layer_ok (read it), and they may call layer_forward(x, W, b). At least ONE of your tests
# must be parametrized with @pytest.mark.parametrize, for example over several shapes:
#
# @pytest.mark.parametrize("n, n_in, n_out", [(1, 3, 2), (5, 4, 4)])
# def test_something(n, n_in, n_out):
#     ...
#     assert ...

my_tests_25 = ...   # the list of your test functions

In [ ]:
def parametrized_25(tests):
    """True when at least one test function carries a pytest.mark.parametrize mark."""
    return any(mark.name == "parametrize" for t in tests for mark in getattr(t, "pytestmark", []))


if my_tests_25 is ...:
    print("⏳ Ex 18.25 : tests pas encore écrits.")
elif not isinstance(my_tests_25, (list, tuple)) or not all(callable(t) for t in my_tests_25):
    print("❌ Ex 18.25 : my_tests_25 doit être la liste de tes fonctions de test.")
else:
    helpers_25 = [layer_forward] + ([numeric_25] if "numeric_25" in globals() else [])
    good_25 = wb.run_pytest(my_tests_25, subject=layer_ok, name="layer_backward", extra=helpers_25)
    verdict("18.25", parametrized_25(my_tests_25), "au moins un de tes tests est paramétré.",
            "aucun de tes tests n'est paramétré : utilise @pytest.mark.parametrize (sur des formes, par exemple).")
    verdict("18.25", good_25.ok, f"tes tests passent sur la version juste ({good_25.passed} cas).",
            "tes tests doivent tous passer sur layer_ok, la version juste (voir le détail au-dessus).")
    if good_25.ok:
        for bug_25 in [layer_bug_1, layer_bug_2, layer_bug_3, layer_bug_4, layer_bug_5]:
            result_25 = wb.run_pytest(my_tests_25, subject=bug_25, name="layer_backward", extra=helpers_25, quiet=True)
            verdict("18.25", result_25.failed + result_25.errors > 0, f"{bug_25.__name__} est attrapé ({result_25.failed} cas en échec).",
                    f"tes tests passent sur {bug_25.__name__}, qui est faux : lis-le, puis ajoute un test qui le fait échouer.")

## Partie D · Le micro-autograd, puis le défi

Écrire un micro-autograd, la classe `Value` (18.26), comparer quatre façons d'obtenir les mêmes gradients (18.27), assembler des `Value` en un petit réseau entraînable, `MiniMLP` (18.28), puis relever le défi MNIST avec ton MLP NumPy (18.29).

### Ex 18.26 — Micro-autograd : la classe Value 🔨 ★★★ ⏱️ 90 min
**Objectif :** écrire un scalaire qui enregistre ses opérations et calcule tous ses gradients en remontant le graphe.  
**Prérequis :** ch. 16 (`topological_order`) · 0B (règle de la chaîne à plusieurs chemins) · 0A (classes) · fiche, section sur le micro-autograd (encadré sur la surcharge d'opérateurs et les fermetures) · **mylearn :** `autograd_mini.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/autograd_mini.py` (créé par `python tools/start_chapter.py 18`), lis la docstring de la fonction, remplace le `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement ; PyTorch et SciPy sont les **oracles** des tests (sans PyTorch, les tests qui en ont besoin sont ignorés). La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à des calculs indépendants, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch18_autograd_mini.py -q` les lance tous dans un terminal.

Écris la classe `Value` : ses opérateurs `+`, `-`, `*`, `/`, `**`, le moins unaire, leurs versions « à droite » (`2 * x`, `1 - x`, `3 / x`), les méthodes `exp`, `log`, `tanh`, `relu`, `sigmoid`, et `backward`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- chaque opération crée une **nouvelle** `Value`, dont `_prev` contient ses parents dans l'ordre (`(a * a)._prev == (a, a)`), et lui attache une fonction `_backward` qui **ajoute** (`+=`) à chaque parent sa part du gradient ;
- un opérande qui n'est ni une `Value` ni un nombre réel (une chaîne, `None`, une liste) lève `TypeError`, comme `x ** Value(...)` : l'exposant de `**` est un nombre ;
- `log` d'une valeur $\le 0$ lève `ValueError` ; $\mathrm{ReLU}'(0) = 0$, comme PyTorch ; `sigmoid(±1000)` ne déborde pas ;
- `backward()` range le graphe dans un ordre topologique en visitant **chaque nœud une seule fois**, met `self.grad` à 1 (sans l'ajouter), puis appelle les `_backward` dans l'ordre inverse ; il ne remet aucun autre gradient à 0 : deux appels de suite additionnent les gradients, comme dans PyTorch ;
- les tests vont jusqu'à des graphes de quelques centaines de niveaux.

La vérification essaie l'exemple de la docstring, compare les trois gradients d'une expression compliquée à des différences finies, essaie un graphe où chaque nœud sert deux fois, des opérandes interdits, puis lance les tests (`-k test_value_`).

Dans tes notes : pourquoi faut-il `+=` et non `=` dans chaque `_backward` ? Que deviendrait le gradient de `a` dans `c = a * b + a` avec `=` ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")   # reload your saved file
with checked("18.26"):
    V = mylearn.autograd_mini.Value
    a_26, b_26 = V(2.0), V(-3.0)
    c_26 = a_26 * b_26 + a_26                    # a is used twice: its gradients add up
    c_26.backward()
    same("18.26", "(c.data, a.grad, b.grad) pour c = a b + a (docstring)", [c_26.data, a_26.grad, b_26.grad], [-4.0, -2.0, 2.0])
    same("18.26", "(1 - a / 4).data", [(1 - a_26 / 4).data], [0.5])
with checked("18.26"):
    V = mylearn.autograd_mini.Value
    point_26 = [0.7, -1.3, 2.1]

    def expression_26(x, y, w):
        """The same expression for Values and for floats (math)."""
        if isinstance(x, float):
            return math.tanh(x * y + math.exp(w / 3)) / (y ** 2 + 1) - 2 / (w + x) + math.log(w)
        return (x * y + (w / 3).exp()).tanh() / (y ** 2 + 1) - 2 / (w + x) + w.log()

    values_26 = [V(v) for v in point_26]
    out_26 = expression_26(*values_26)
    out_26.backward()
    numeric_26 = []
    for k_26 in range(3):
        plus_26, minus_26 = list(point_26), list(point_26)
        plus_26[k_26] += 1e-6
        minus_26[k_26] -= 1e-6
        numeric_26.append((expression_26(*plus_26) - expression_26(*minus_26)) / 2e-6)
    same("18.26", "la valeur de tanh(xy + e^(w/3)) / (y² + 1) - 2 / (w + x) + log w", [out_26.data], [expression_26(*point_26)])
    same("18.26", "ses trois gradients contre les différences finies", [v.grad for v in values_26], numeric_26, rtol=1e-6, atol=1e-8)
    x_26 = V(0.5)
    y_26 = x_26
    for _ in range(12):
        y_26 = y_26 * 0.5 + y_26 * 0.5           # each level uses the previous one twice
    y_26.backward()
    verdict("18.26", abs(x_26.grad - 1.0) < 1e-12, "un graphe de 12 niveaux où chaque nœud sert deux fois : x.grad vaut 1.",
            f"un graphe de 12 niveaux où chaque nœud sert deux fois : x.grad vaut {x_26.grad!r}, attendu 1 "
            "(chaque nœud doit être visité une seule fois, et ses gradients s'additionner).")
    refuses("18.26", "Value(1.0) + \"a\"", lambda: V(1.0) + "a", TypeError)
    refuses("18.26", "Value(-1.0).log()", lambda: V(-1.0).log(), ValueError)
if written(lambda: mylearn.autograd_mini.Value(1.0) + 1.0):
    run_mylearn_tests("autograd_mini", keyword="test_value_")
else:
    print("⏳ Ex 18.26 : les tests attendent ta classe.")

### Ex 18.27 — À la main, Value, mlp_backward, différences finies : quatre gradients identiques ? 🔮 ★★ ⏱️ 20 min
**Objectif :** prédire de combien diffèrent quatre façons de calculer les mêmes gradients, et ce qui se passe sur un coude de ReLU.  
**Prérequis :** Ex 18.26 · Ex 18.18 · ✏️ 18.4 · fiche, sections sur la vérification d'un gradient et sur le micro-autograd · **Parcours :** M, C

Reprends le réseau de ✏️ 18.4 (deux entrées, deux neurones cachés sans activation, deux sorties linéaires, sans biais, $\mathbf{x} = (1 ;\ 2)$, label $(1 ;\ 0)$, $L = \frac12\sum (\hat{y}_k - y_k)^2$). `four_gradients_27` (fournie) calcule les gradients de $\mathbf{W}_1$ et $\mathbf{W}_2$ par **ton** `mlp_backward` (sur le cache de ton `mlp_forward`, avec `dout = out − y`), par **ta** classe `Value`, et par des différences finies centrées ($\varepsilon = 10^{-6}$) ; l'expérience les compare aussi à tes réponses de 18.4 f et g (partie 0), si tu les as remplies.

Prédis, **avant** d'exécuter :
a) `exact_27` : le plus grand écart entre `mlp_backward` et `Value` : (A) exactement 0 ; (B) non nul, mais moins de $10^{-12}$ ; (C) entre $10^{-12}$ et $10^{-6}$ ; (D) plus de $10^{-6}$ ;
b) `finite_27` : le plus grand écart entre `mlp_backward` et les différences finies, avec les mêmes choix ;
c) `kink_27` : on donne maintenant une ReLU aux neurones cachés, et l'on change un poids pour que la pré-activation de $A$ vaille **exactement** 0. L'erreur relative (la mesure de `gradient_check`) entre `mlp_backward` et les différences finies sur $\mathbf{W}_1$ : (A) moins de $10^{-7}$ ; (B) environ $\frac13$ ; (C) environ 1.

Puis exécute l'expérience : elle affiche les écarts entre les méthodes (et avec tes réponses de 18.4, si tu les as remplies), l'expérience du coude, et vérifie tes prédictions.

In [ ]:
X_27 = np.array([[1.0, 2.0]])                       # the example of ✏️ 18.4, its label (1, 0)
Y_27 = np.array([[1.0, 0.0]])
W1_27 = np.array([[0.5, 1.0], [-0.5, 0.25]])          # rows X, Y ; columns A, B
W2_27 = np.array([[1.0, -1.0], [0.5, 0.5]])           # rows A, B ; columns C, D


def four_gradients_27(W1, W2, hidden="identity"):
    """The gradients of L = 1/2 sum (out - y)^2 with respect to W1 and W2 (no bias), by three methods.

    Returns {"mlp_backward": [dW1, dW2], "Value": [dW1, dW2], "differences": [dW1, dW2]}: YOUR mlp_backward on the
    cache of YOUR mlp_forward, YOUR Value, and central differences (eps = 1e-6) computed here with NumPy.
    hidden = "identity" or "relu" (the activation of A and B; the outputs C and D stay linear).
    """
    A, L, B, V = mylearn.nn.activations, mylearn.nn.layers, mylearn.nn.backward, mylearn.autograd_mini.Value
    f, df = (None, None) if hidden == "identity" else (A.relu, A.relu_derivative)
    params = [(W1.copy(), np.zeros(2)), (W2.copy(), np.zeros(2))]
    out, cache = L.mlp_forward(X_27, params, f, None, return_cache=True)
    by_backward = [dW for dW, _ in B.mlp_backward(out - Y_27, cache, params, activation_derivative=df)]

    w1 = [[V(float(v)) for v in row] for row in W1]
    w2 = [[V(float(v)) for v in row] for row in W2]
    hidden_values = [V(0.0) + X_27[0, 0] * w1[0][j] + X_27[0, 1] * w1[1][j] for j in range(2)]
    if hidden == "relu":
        hidden_values = [h.relu() for h in hidden_values]
    outputs = [hidden_values[0] * w2[0][k] + hidden_values[1] * w2[1][k] for k in range(2)]
    loss = 0.5 * ((outputs[0] - Y_27[0, 0]) ** 2 + (outputs[1] - Y_27[0, 1]) ** 2)
    loss.backward()
    by_value = [np.array([[v.grad for v in row] for row in w]) for w in (w1, w2)]

    def numpy_loss(M1, M2):
        h = X_27 @ M1
        h = h if hidden == "identity" else np.maximum(h, 0.0)
        return 0.5 * float(np.sum((h @ M2 - Y_27) ** 2))

    M1, M2 = W1.copy(), W2.copy()
    by_differences = [numeric_grad(lambda: numpy_loss(M1, M2), M) for M in (M1, M2)]
    return {"mlp_backward": by_backward, "Value": by_value, "differences": by_differences}


def gap_27(first, second):
    """The largest absolute gap between two lists [dW1, dW2]."""
    return float(max(np.max(np.abs(np.asarray(a) - np.asarray(b))) for a, b in zip(first, second)))

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
exact_27 = ...    # a) "A", "B", "C" or "D"
finite_27 = ...   # b) "A", "B", "C" or "D"
kink_27 = ...     # c) "A", "B" or "C"

In [ ]:
if any(answer is ... or answer is None for answer in [exact_27, finite_27, kink_27]):
    print("⏳ Ex 18.27 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    with wb.attempt("18.27"):
        grads_27 = four_gradients_27(W1_27, W2_27)
        if filled(globals().get("answer_18_4f"), globals().get("answer_18_4g")):
            try:
                hand_27 = [np.reshape(np.asarray(answer_18_4g, dtype=float), (2, 2)), np.reshape(np.asarray(answer_18_4f, dtype=float), (2, 2))]
                print(f"{'by hand':13s} (your answers to 18.4 g and f, part 0) · largest gap with mlp_backward: "
                      f"{gap_27(hand_27, grads_27['mlp_backward']):.3g}")
            except (TypeError, ValueError):
                print("(tes réponses à 18.4 f et g, en partie 0, ne sont pas des listes de 4 nombres : comparaison à la main sautée)")
        print(f"largest gap mlp_backward / Value: {gap_27(grads_27['mlp_backward'], grads_27['Value']):.3g} · "
              f"mlp_backward / differences: {gap_27(grads_27['mlp_backward'], grads_27['differences']):.3g}")
        W1_kink_27 = W1_27.copy()
        W1_kink_27[0, 0] = 1.0                               # z_A = 1 x 1 + 2 x (-0.5) = 0 exactly: on the kink of ReLU
        kink_grads_27 = four_gradients_27(W1_kink_27, W2_27, hidden="relu")
        print("ReLU hidden layer, z_A = 0 · dW1 by mlp_backward:", np.round(kink_grads_27["mlp_backward"][0], 6).tolist(),
              "· by differences:", np.round(kink_grads_27["differences"][0], 6).tolist())
        kink_error_27 = relative_error(kink_grads_27["mlp_backward"][0], kink_grads_27["differences"][0])
        print(f"relative error on dW1 (the measure of gradient_check): {kink_error_27:.3g}")
    wb.check("18.27a", exact_27)
    wb.check("18.27b", finite_27)
    wb.check("18.27c", kink_27)

Dans tes notes : d'où vient l'écart des différences finies, et pourquoi a-t-il cet ordre de grandeur ? Sur le coude, quelle valeur donne la différence centrée pour les poids qui arrivent dans $A$, et pourquoi ? Que dit l'expérience sur la façon de tester un gradient de ReLU ?

### Ex 18.28 — MiniMLP : entraîner un petit réseau fait de Value 🔨 ★★★ ⏱️ 45 min
**Objectif :** assembler des Value en neurones et en couches, puis entraîner ce petit réseau sur les lunes.  
**Prérequis :** Ex 18.26 · ch. 16 (`mlp_forward`, formes des poids) · fiche §18.9, §18.12.1, section sur le micro-autograd · **Fil rouge :** synth · **mylearn :** `autograd_mini.py` · **Parcours :** C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/autograd_mini.py` (créé par `python tools/start_chapter.py 18`), lis la docstring de la fonction, remplace le `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard seulement ; PyTorch et SciPy sont les **oracles** des tests (sans PyTorch, les tests qui en ont besoin sont ignorés). La cellule de vérification recharge ta librairie, essaie les exemples de la docstring, compare quelques résultats à des calculs indépendants, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch18_autograd_mini.py -q` les lance tous dans un terminal.

Le constructeur de `MiniMLP(n_in, layer_sizes, activation="relu", random_state=None)` est fourni : il crée les poids (`weights_`, une liste de matrices de `Value` de forme `n_in × n_out` par couche) et les biais (`biases_`). Écris ses quatre méthodes. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- `model(x)` reçoit une suite de `n_in` nombres ou `Value` (une autre longueur lève `ValueError`), calcule $z_j = b_j + \sum_i x_i W_{ij}$ couche par couche, applique `activation` aux couches cachées (`"linear"` : aucune), laisse la dernière couche **linéaire**, et rend une `Value` seule si la dernière couche a une sortie, une liste sinon ;
- `parameters()` rend les `Value` **elles-mêmes** (pas des copies), poids de chaque couche ligne par ligne puis ses biais, couche après couche ;
- `zero_grad()` remet à 0 le gradient de **tous** les paramètres, biais compris ;
- `to_params()` rend des **copies** `float64` lues au moment de l'appel : la liste des `(W, b)` de formes `(n_in, n_out)` et `(n_out,)`, prête pour `mlp_forward`.

La vérification compare les sorties d'un `MiniMLP` 3-5-4-2 à ton `mlp_forward` sur `to_params()`, vérifie `zero_grad`, entraîne un `MiniMLP` 2-8-1 (tanh) sur 100 lunes par `train_28` (fournie : la loss logistique moyenne, $\log(1 + e^{-s\,\hat{y}})$ pour un label $s = \pm 1$, et une descente de gradient sur tout le batch ; une dizaine de secondes, car chaque nombre est un objet Python), trace sa loss et sa frontière, puis lance les tests (`-k test_mini_mlp_`).

Dans tes notes : combien de `Value` le graphe d'un pas d'entraînement contient-il environ ? Combien de temps prendrait le même pas avec `mlp_forward` et `mlp_backward` ? Pourquoi PyTorch construit-il son graphe sur des tenseurs plutôt que sur des nombres ?

In [ ]:
N_28, STEPS_28 = (100, 60) if FAST_MODE else (200, 150)
X_28, y_28 = wb.synth.make_moons(n=N_28, noise=0.1, seed=1828)
X_28 = (X_28 - X_28.mean(axis=0)) / X_28.std(axis=0)
SIGNS_28 = 2.0 * y_28 - 1.0                           # labels -1 and +1


def train_28(model, steps, lr=0.5):
    """Full-batch gradient descent of a MiniMLP with ONE output on the logistic loss mean log(1 + exp(-s * out)),
    s = -1 or +1. Returns the loss before each step (floats)."""
    losses = []
    for _ in range(steps):
        total = None
        for x, s in zip(X_28, SIGNS_28):
            term = ((-s) * model([float(v) for v in x])).exp() + 1.0
            total = term.log() if total is None else total + term.log()
        loss = total / len(X_28)
        model.zero_grad()
        loss.backward()
        for p in model.parameters():
            p.data -= lr * p.grad
        losses.append(loss.data)
    return losses

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="18")   # reload your saved file
with checked("18.28"):
    M = mylearn.autograd_mini.MiniMLP
    model_28 = M(2, [3, 1], activation="tanh", random_state=0)
    verdict("18.28", [W.shape for W, b in model_28.to_params()] == [(2, 3), (3, 1)] and len(model_28.parameters()) == 13,
            "MiniMLP(2, [3, 1]) : formes [(2, 3), (3, 1)] et 13 paramètres, comme la docstring.",
            "MiniMLP(2, [3, 1]) : to_params() doit rendre des W de formes [(2, 3), (3, 1)], et parameters() 13 Value.")
    model_28 = M(3, [5, 4, 2], activation="tanh", random_state=1828)
    x_28 = [0.3, -1.2, 0.8]
    outputs_28 = model_28(x_28)
    mine_28 = [o.data for o in outputs_28] if isinstance(outputs_28, list) else None
    want_28 = mylearn.nn.layers.mlp_forward(np.array([x_28]), model_28.to_params(), np.tanh)[0]
    if mine_28 is None:
        print(f"❌ Ex 18.28 : un MiniMLP à 2 sorties doit rendre une liste de 2 Value, reçu {type(outputs_28).__name__}.")
    else:
        same("18.28", "les sorties d'un MiniMLP 3-5-4-2 contre mlp_forward sur to_params()", mine_28, want_28)
    outputs_28[0].backward() if isinstance(outputs_28, list) else None
    model_28.zero_grad()
    verdict("18.28", all(p.grad == 0.0 for p in model_28.parameters()), "zero_grad remet tous les gradients à 0.",
            "après zero_grad, au moins un paramètre a encore un gradient non nul.")
CURVE_28 = None
if written(lambda: mylearn.autograd_mini.MiniMLP(2, [1], random_state=0)([0.0, 0.0])):
    with checked("18.28"):
        net_28 = mylearn.autograd_mini.MiniMLP(2, [8, 1], activation="tanh", random_state=1828)
        started_28 = time.perf_counter()
        CURVE_28 = train_28(net_28, STEPS_28)
        print(f"{STEPS_28} steps on {N_28} points in {time.perf_counter() - started_28:.1f} s · loss {CURVE_28[0]:.3f} -> {CURVE_28[-1]:.3f}")
        W_28 = net_28.to_params()
        acc_28 = float(np.mean((mylearn.nn.layers.mlp_forward(X_28, W_28, np.tanh)[:, 0] > 0) == y_28))
        verdict("18.28", CURVE_28[-1] < 0.5 * CURVE_28[0] and acc_28 >= 0.85,
                f"le MiniMLP apprend : loss divisée par {CURVE_28[0] / CURVE_28[-1]:.1f}, accuracy {acc_28:.3f}.",
                f"le MiniMLP n'apprend pas assez (loss {CURVE_28[0]:.3f} -> {CURVE_28[-1]:.3f}, accuracy {acc_28:.3f}) : "
                "vérifie parameters() (les Value elles-mêmes) et le signe de la mise à jour.")
        fig_28, (ax_c, ax_b) = plt.subplots(1, 2, figsize=(11, 3.6))
        ax_c.plot(CURVE_28)
        ax_c.set_xlabel("pas")
        ax_c.set_ylabel("loss logistique")
        wb.plot.plot_decision_boundary(type("Net", (), {"predict": lambda self, X: (mylearn.nn.layers.mlp_forward(X, W_28, np.tanh)[:, 0] > 0).astype(int)})(),
                                       X_28, y_28, ax=ax_b, title="frontière du MiniMLP")
        plt.tight_layout()
        plt.show()
    run_mylearn_tests("autograd_mini", keyword="test_mini_mlp_")
else:
    print("⏳ Ex 18.28 : les tests attendent ta classe.")

### Ex 18.29 — Défi MNIST : au moins 97 % avec ton MLP NumPy 🏆 ★★★★ ⏱️ 120 min
**Objectif :** entraîner un MLP avec ta propre librairie, sans PyTorch ni scikit-learn, jusqu'à 97 % d'accuracy sur le test de MNIST.  
**Prérequis :** Ex 18.19 · Ex 18.14 · ch. 16 (initialisations) · fiche §18.11, §18.12.5 · **Fil rouge :** MNIST · **Parcours :** C

Les 60 000 images d'entraînement de MNIST (`X_TRAIN_29`, valeurs dans $[0, 1]$, et leurs labels), et un **test caché** de 10 000 images. Écris `train_29(X, y)`, qui entraîne un MLP avec **ta** librairie (`init_mlp`, `mlp_forward`, une loss de 18.14, `mlp_backward` ; NumPy, aucune autre bibliothèque d'apprentissage) et rend le couple `(params, activation)` : la liste des `(W, b)` (784 entrées, 10 sorties, les logits) et la fonction d'activation cachée que `mlp_forward` doit utiliser. Tu choisis l'architecture, l'initialisation, la loss, le learning rate et son évolution, la taille des mini-batches et le nombre d'epochs.

Règles : l'entraînement dure **au plus 3 minutes** sur ta machine (la vérification le mesure) ; le défi est réussi avec une accuracy d'au moins **97 %** sur le test caché ; palier 🌟 : au moins 98 %. Règle d'honneur : n'utilise pas le test caché pour régler ta méthode (garde, par exemple, les 10 000 dernières images d'entraînement comme validation pendant tes essais) ; la vérification ne le révèle qu'une fois par session, et fige le résultat.

Dans tes notes : ta démarche (ce que tu as essayé, dans quel ordre, avec quelle validation), le temps d'une epoch, et ce qui a le plus compté : la taille du réseau, le learning rate, la taille des batches ou le nombre d'epochs ?

In [ ]:
X_TRAIN_29, Y_TRAIN_29 = wb.datasets.load_mnist("train", flatten=True, normalize=True)
X_HIDDEN_29, Y_HIDDEN_29 = wb.datasets.load_mnist("test", flatten=True, normalize=True)   # the hidden test: do not look at it
TIME_LIMIT_29 = 180.0                                  # seconds of training, on your machine
print("train:", X_TRAIN_29.shape, X_TRAIN_29.dtype, "· hidden test:", X_HIDDEN_29.shape)

In [ ]:
def train_29(X, y):
    """Train an MLP with YOUR library (mylearn.nn: init_mlp, mlp_forward, a loss of 18.14, mlp_backward; NumPy only)
    on the images X (shape (60000, 784), values in [0, 1]) and their labels y.

    Returns (params, activation): the trained [(W, b), ...] (784 inputs, 10 outputs: the logits) and the hidden
    activation function used by mlp_forward (for example mylearn.nn.activations.relu).
    """
    raise NotImplementedError("train_29")

In [ ]:
with checked("18.29"):
    started_29 = time.perf_counter()
    result_29 = train_29(X_TRAIN_29, Y_TRAIN_29)
    seconds_29 = time.perf_counter() - started_29
    ok_29 = isinstance(result_29, tuple) and len(result_29) == 2 and callable(result_29[1])
    if not ok_29:
        print("❌ Ex 18.29 : train_29 doit rendre le tuple (params, activation), activation étant une fonction.")
    else:
        params_29, activation_29 = result_29
        shapes_29 = [np.shape(W) for W, _ in params_29] if isinstance(params_29, list) else []
        ok_29 = (len(shapes_29) >= 2 and shapes_29[0][0] == 784 and shapes_29[-1][1] == 10
                 and all(a[1] == b[0] for a, b in zip(shapes_29[:-1], shapes_29[1:])))
        verdict("18.29", ok_29, f"un MLP {[784] + [s[1] for s in shapes_29]}, entraîné en {seconds_29:.0f} s.",
                f"params doit être une liste de (W, b) qui va de 784 entrées à 10 sorties ; reçu les formes {shapes_29}.")
        verdict("18.29", seconds_29 <= TIME_LIMIT_29, f"entraînement en moins de {TIME_LIMIT_29:.0f} s.",
                f"entraînement en {seconds_29:.0f} s : le défi demande moins de {TIME_LIMIT_29:.0f} s (moins d'epochs, un réseau plus petit, des batches plus grands).")
        ok_29 = ok_29 and seconds_29 <= TIME_LIMIT_29
    if ok_29:
        train_acc_29 = float(np.mean(mylearn.nn.layers.mlp_forward(X_TRAIN_29[:10_000], params_29, activation_29).argmax(axis=1)
                                     == Y_TRAIN_29[:10_000]))
        print(f"accuracy on 10 000 training images: {train_acc_29:.4f}")
        if globals().get("REVEALED_29") is None:              # revealed once per session (an honour rule)
            hidden_acc_29 = float(np.mean(mylearn.nn.layers.mlp_forward(X_HIDDEN_29, params_29, activation_29).argmax(axis=1) == Y_HIDDEN_29))
            REVEALED_29 = (hidden_acc_29, [784] + [s[1] for s in shapes_29], round(seconds_29))
        test_29, sizes_29, time_29 = REVEALED_29
        print(f"hidden test: {test_29:.4f} (revealed once per session, for the network {sizes_29} trained in {time_29} s)")
        verdict("18.29", test_29 >= 0.97,
                "défi réussi : au moins 97 % sur les 10 000 images du test caché." + (" 🌟 Palier atteint : au moins 98 %." if test_29 >= 0.98 else ""),
                "pas encore : il faut au moins 97 % sur le test caché.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dérouler à la main la passe avant, les deltas et un pas de mise à jour d'un petit réseau ?
2. Sais-tu écrire les gradients d'une couche dense pour un mini-batch, et vérifier leurs formes ?
3. Sais-tu vérifier un gradient par différences finies, et reconnaître un learning rate mal réglé ?

**Pour aller plus loin** : le chapitre 2 du livre de M. Nielsen, les notes de CS231n et micrograd d'A. Karpathy, cités dans la fiche. La suite : le ch. 19, où les optimiseurs remplaceront la mise à jour $w \leftarrow w - \eta g$.